<a href="https://colab.research.google.com/github/ornab74/pufferforge/blob/main/SurfGuard_USA_PufferForge_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SurfGuard-USA: Continental U.S. Beach Hazard Intelligence

**Google Colab pipeline for historical coastal-hazard collection, supervised learning, PufferForge reinforcement learning, probabilistic hot-zone forecasting, configurable OpenAI Responses API warning generation, and audio alerts.**

This notebook is restricted to the ocean-facing coasts of the **contiguous United States**:

- Atlantic, Gulf of Mexico, and Pacific coasts
- Excludes Alaska, Hawaii, Puerto Rico, other territories, and Great Lakes coastlines

## What the system predicts

The model estimates the probability that conditions resemble historically dangerous coastal episodes and recommends an alert action. It does **not** claim that a specific person will require rescue at an exact future time.

| Horizon | Output | Scientific interpretation |
|---|---|---|
| 0–16 days | Operational risk forecast | Harmonic tide predictions plus optional current NOAA/NCEP wave guidance |
| 17–300 days | Planning-risk calendar | Exact astronomical tide timing plus seasonal wave/hazard climatology and widened uncertainty |
| Historical | Hazard reconstruction | NOAA incident reports, tide observations/predictions, buoy measurements, beach coordinates, and optional rescue logs |

> **Safety boundary:** This research notebook is not an operational warning authority, navigation product, or substitute for NOAA/NWS alerts, beach closures, lifeguards, emergency managers, or local observations. Never automate a public closure or lifesaving instruction without human review and official-source confirmation.

## System architecture

\[
x_{b,t} =
[\text{tide},\ \Delta\text{tide},\ \text{surge residual},\ H_s,\ T_p,\ \theta,\ U,\ \text{season},\ \text{crowd proxies},\ \text{site}]
\]

A calibrated ensemble estimates:

\[
p_{b,t}=P(\text{dangerous coastal incident}\mid x_{b,t})
\]

A PufferForge PPO policy then chooses an alert action \(a_t\in\{0,1,2\}\):

- **0 — Normal monitoring**
- **1 — Advisory / enhanced observation**
- **2 — Strong warning / closure review**

The RL reward penalizes missed dangerous episodes much more heavily than false alarms. The configured OpenAI model receives only grounded model outputs and converts them into concise, human-readable warnings. GPT does not replace the numerical forecast model and does not perform the PPO backpropagation.

## 1. Runtime installation

In [ ]:
# Install the scientific stack and PufferForge.
# Re-running this cell is safe. Native compilation is attempted first. If
# packaging or compilation fails, the source-tree Python runtime remains usable.

import os
import sys
import subprocess
from pathlib import Path

BUILD_PACKAGES = [
    "pip>=24",
    "setuptools>=61",
    "wheel>=0.44",
    "pybind11>=2.13",
]

CORE_PACKAGES = [
    "numpy>=1.26",
    "pandas>=2.2",
    "pyarrow>=16",
    "duckdb>=1.0",
    "requests>=2.32",
    "requests-cache>=1.2",
    "tenacity>=8.3",
    "beautifulsoup4>=4.12",
    "lxml>=5.2",
    "tqdm>=4.66",
    "scikit-learn>=1.5",
    "joblib>=1.4",
    "holidays>=0.52",
    "folium>=0.17",
    "plotly>=5.22",
    "openai>=1.68",
    "xarray>=2024.6",
    "soundfile>=0.12",
]

# Required only when RUN_OPERATIONAL_GFSWAVE=True.
OPTIONAL_GRIB_PACKAGES = ["cfgrib>=0.9.14", "eccodes>=2.37"]


def run_command(cmd, desc, *, env=None, required=False, quiet=False):
    print(f"--- {desc} ---")
    completed = subprocess.run(
        cmd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    output = completed.stdout or ""
    if completed.returncode == 0:
        if output.strip() and not quiet:
            print(output.rstrip())
        return True

    print(f"Command failed with exit code {completed.returncode}: {' '.join(map(str, cmd))}")
    # Always surface the useful tail instead of hiding pip's real error behind -q.
    lines = output.rstrip().splitlines()
    print("\n".join(lines[-120:]) if lines else "(no subprocess output)")
    if required:
        raise RuntimeError(f"Required setup step failed: {desc}")
    return False


run_command(
    [sys.executable, "-m", "pip", "install", "--upgrade", *BUILD_PACKAGES],
    "Updating Python build tooling",
    required=False,
    quiet=True,
)
run_command(
    [sys.executable, "-m", "pip", "install", *CORE_PACKAGES],
    "Installing core scientific packages",
    required=True,
    quiet=True,
)

try:
    import torch  # noqa: F401
except ImportError:
    run_command(
        [sys.executable, "-m", "pip", "install", "torch>=2.2"],
        "Installing PyTorch",
        required=True,
    )

optional_status = {}
for package in OPTIONAL_GRIB_PACKAGES:
    optional_status[package.split(">=")[0]] = run_command(
        [sys.executable, "-m", "pip", "install", package],
        f"Installing optional package {package}",
        quiet=True,
    )

repo_dir = Path("/content/pufferforge")
if repo_dir.exists() and not (repo_dir / ".git").exists():
    import shutil
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    run_command(
        ["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/ornab74/pufferforge", str(repo_dir)],
        "Cloning PufferForge repository",
        required=True,
    )
else:
    # A Colab checkout is disposable. Resetting avoids stale local build files or
    # a previous interrupted notebook run blocking a fast-forward pull.
    refreshed = run_command(
        ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"],
        "Refreshing PufferForge repository",
    )
    if refreshed:
        run_command(
            ["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"],
            "Resetting PufferForge checkout to origin/main",
            required=True,
        )

pufferforge_available = False
pufferforge_native_available = False
install_env = os.environ.copy()
install_env["PUFFERFORGE_BUILD_NATIVE"] = "1"

# Use the runtime toolchain directly. The repository build requirements are
# intentionally minimal, so this also works when an isolated package index
# cannot resolve a newer setuptools wheel.
native_install = run_command(
    [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--editable", str(repo_dir)],
    "Installing PufferForge with native extension",
    env=install_env,
)

fallback_install = False
if not native_install:
    fallback_env = install_env.copy()
    fallback_env["PUFFERFORGE_BUILD_NATIVE"] = "0"
    fallback_install = run_command(
        [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--editable", str(repo_dir)],
        "Installing PufferForge pure-Python package",
        env=fallback_env,
    )

# This source-tree path is the final deterministic fallback. PPOTrainer and
# PythonVectorEnv do not require the optional C++ extension.
source_dir = str(repo_dir / "python")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

try:
    import importlib
    if "pufferforge" in sys.modules:
        importlib.reload(sys.modules["pufferforge"])
    import pufferforge
    from pufferforge import PPOTrainer, PythonVectorEnv, TrainConfig
    pufferforge_available = True
    pufferforge_native_available = bool(getattr(pufferforge, "NATIVE_AVAILABLE", False))
except Exception as exc:
    print("PufferForge import failed after native, package-fallback, and source-tree setup.")
    print(f"{type(exc).__name__}: {exc}")

if not pufferforge_available:
    raise RuntimeError("PufferForge could not be imported; inspect the full pip output above.")

print({
    "pufferforge_available": pufferforge_available,
    "pufferforge_native_available": pufferforge_native_available,
    "native_install": native_install,
    "fallback_install": fallback_install,
    "source_tree_fallback": not native_install and not fallback_install,
    "optional_packages": optional_status,
})

## 2. Configuration and storage

In [ ]:
from __future__ import annotations

import os
import re
import io
import gc
import json
import math
import time
import gzip
import shutil
import random
import hashlib
import warnings
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Iterable, Sequence, Any
from urllib.parse import urljoin

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# ---------- Scale controls ----------
# False: compact demonstration centered on incident-associated station-years.
# True: partitioned national collection. Expect many requests and substantial Drive usage.
FULL_SCALE = False

HISTORICAL_START_YEAR = 1996
HISTORICAL_END_YEAR = int(os.environ.get("SURFGUARD_HISTORICAL_END_YEAR", datetime.now(timezone.utc).year - 1))
FORECAST_DAYS = 300
OPERATIONAL_WAVE_DAYS = 16
COOPS_MAX_REQUEST_DAYS = 31

DEMO_MAX_STATION_YEARS = int(os.environ.get("SURFGUARD_DEMO_MAX_STATION_YEARS", "12"))
FULL_MAX_STATION_YEARS = None
NEGATIVE_TO_POSITIVE_RATIO = 20
LABEL_WINDOW_HOURS = 3
MAX_EVENT_TO_STATION_KM = 175
MAX_BEACH_TO_STATION_KM = 100
MAX_STATION_TO_BUOY_KM = 350
RANDOM_SEED = 56

# OpenAI API model IDs are configurable because account access can differ.
# The default follows the official Responses API quickstart; override with an
# environment variable without editing the notebook.
OPENAI_MODEL = os.environ.get("SURFGUARD_OPENAI_MODEL", "gpt-5")
OPENAI_TTS_MODEL = os.environ.get("SURFGUARD_OPENAI_TTS_MODEL", "gpt-4o-mini-tts")
OPENAI_TTS_VOICE = os.environ.get("SURFGUARD_OPENAI_TTS_VOICE", "alloy")

# Ocean-facing contiguous states only.
COASTAL_STATES = [
    "ME", "NH", "MA", "RI", "CT", "NY", "NJ", "DE", "MD", "VA",
    "NC", "SC", "GA", "FL", "AL", "MS", "LA", "TX",
    "CA", "OR", "WA",
]
STATE_NAMES = {
    "ME": "MAINE", "NH": "NEW HAMPSHIRE", "MA": "MASSACHUSETTS",
    "RI": "RHODE ISLAND", "CT": "CONNECTICUT", "NY": "NEW YORK",
    "NJ": "NEW JERSEY", "DE": "DELAWARE", "MD": "MARYLAND",
    "VA": "VIRGINIA", "NC": "NORTH CAROLINA", "SC": "SOUTH CAROLINA",
    "GA": "GEORGIA", "FL": "FLORIDA", "AL": "ALABAMA",
    "MS": "MISSISSIPPI", "LA": "LOUISIANA", "TX": "TEXAS",
    "CA": "CALIFORNIA", "OR": "OREGON", "WA": "WASHINGTON",
}
STATE_NAME_TO_ABBR = {v: k for k, v in STATE_NAMES.items()}

DANGEROUS_EVENT_TYPES = {
    "Rip Current",
    "High Surf",
    "Sneakerwave",
    "Storm Surge/Tide",
    "Coastal Flood",
    "Astronomical Low Tide",
    "Tsunami",
    "Marine High Wind",
    "Marine Strong Wind",
    "Hurricane (Typhoon)",
    "Tropical Storm",
}

# Optional local data. Supply a CSV with:
# event_time,end_time,beach_name,state,latitude,longitude,rescued_count,
# injuries,deaths,source_url,verified,narrative
CUSTOM_RESCUE_REPORTS_CSV = None

# Use Google Drive when available.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/surfguard_usa")
except Exception:
    ROOT = Path("/content/surfguard_usa")

RAW = ROOT / "raw"
PROCESSED = ROOT / "processed"
MODELS = ROOT / "models"
FORECASTS = ROOT / "forecasts"
REPORTS = ROOT / "reports"
AUDIO = ROOT / "audio"
CACHE = ROOT / "http_cache"

for p in [RAW, PROCESSED, MODELS, FORECASTS, REPORTS, AUDIO, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(RANDOM_SEED)
random.seed(RANDOM_SEED)

print({
    "root": str(ROOT),
    "full_scale": FULL_SCALE,
    "historical_years": [HISTORICAL_START_YEAR, HISTORICAL_END_YEAR],
    "forecast_days": FORECAST_DAYS,
})

### Runtime self-check

This cell validates the repaired package API and distinguishes required from optional dependencies before the network collection begins.

In [ ]:
import importlib.util

required_modules = [
    "numpy", "pandas", "pyarrow", "requests", "requests_cache", "tenacity",
    "sklearn", "folium", "openai", "pufferforge",
]
missing_required = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_required:
    raise RuntimeError(f"Missing required modules after installation: {missing_required}")

from pufferforge import PPOTrainer, PythonVectorEnv, TrainConfig

runtime_status = {
    "pufferforge_version": getattr(pufferforge, "__version__", "unknown"),
    "pufferforge_native": bool(getattr(pufferforge, "NATIVE_AVAILABLE", False)),
    "openai_model": OPENAI_MODEL,
    "openai_tts_model": OPENAI_TTS_MODEL,
    "coops_max_request_days": COOPS_MAX_REQUEST_DAYS,
    "optional_cfgrib": importlib.util.find_spec("cfgrib") is not None,
    "optional_eccodes": importlib.util.find_spec("eccodes") is not None,
}
print(runtime_status)

## 3. Source manifest and reproducibility

In [ ]:
SOURCE_MANIFEST = pd.DataFrame([
    {
        "source": "NOAA/NCEI Storm Events",
        "purpose": "Historical hazardous event labels and narratives",
        "endpoint": "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/",
        "license_note": "U.S. government/open data; retain provenance",
    },
    {
        "source": "NOAA CO-OPS Data API",
        "purpose": "Observed water levels and harmonic tide predictions",
        "endpoint": "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter",
        "license_note": "Official NOAA service; obey request limits",
    },
    {
        "source": "NOAA CO-OPS Metadata API",
        "purpose": "Tide-prediction station coordinates and metadata",
        "endpoint": "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations.json",
        "license_note": "Official NOAA service",
    },
    {
        "source": "NOAA NDBC",
        "purpose": "Historical buoy wind and wave observations",
        "endpoint": "https://www.ndbc.noaa.gov/data/historical/stdmet/",
        "license_note": "Official NOAA service",
    },
    {
        "source": "USGS GNIS / The National Map",
        "purpose": "Named beach coordinates for the contiguous coastal states",
        "endpoint": "https://carto.nationalmap.gov/arcgis/rest/services/geonames/MapServer/5/query",
        "license_note": "U.S. government/public domain",
    },
    {
        "source": "NCEP NOMADS GFS Wave",
        "purpose": "Optional near-term wave guidance, not the 300-day tier",
        "endpoint": "https://nomads.ncep.noaa.gov/cgi-bin/filter_gfswave.pl",
        "license_note": "Official NOAA service; pause at least 10 seconds between looped requests",
    },
    {
        "source": "PufferForge",
        "purpose": "Vectorized PPO alert-policy optimization",
        "endpoint": "https://github.com/ornab74/pufferforge",
        "license_note": "MIT; independently written PufferLib-style architecture",
    },
])

SOURCE_MANIFEST.to_csv(REPORTS / "source_manifest.csv", index=False)
display(SOURCE_MANIFEST)

## 4. Cached HTTP and geospatial helpers

In [ ]:
import requests
import requests_cache
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from sklearn.neighbors import BallTree

SESSION = requests_cache.CachedSession(
    str(CACHE / "requests"),
    backend="sqlite",
    expire_after=timedelta(days=7),
)
SESSION.headers.update({
    "User-Agent": "SurfGuard-USA research notebook; contact via repository ornab74/pufferforge"
})

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=30),
    retry=retry_if_exception_type((requests.RequestException, ValueError)),
    reraise=True,
)
def get_bytes(url: str, params: dict | None = None, timeout: int = 120) -> bytes:
    response = SESSION.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.content

def get_json(url: str, params: dict | None = None, timeout: int = 120) -> dict:
    content = get_bytes(url, params=params, timeout=timeout)
    payload = json.loads(content.decode("utf-8"))
    if isinstance(payload, dict) and payload.get("error"):
        raise ValueError(payload["error"])
    return payload



def require_frame(frame: pd.DataFrame, name: str, columns: Sequence[str] = ()) -> pd.DataFrame:
    """Fail at the source with an actionable message instead of a later KeyError."""
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{name} is missing required columns: {missing}")
    if frame.empty:
        raise RuntimeError(
            f"{name} returned no rows. Check the upstream service, cached files, "
            "geographic filters, and the selected historical years."
        )
    return frame


def date_chunks(start: pd.Timestamp, end: pd.Timestamp, max_days: int = COOPS_MAX_REQUEST_DAYS):
    """Yield inclusive date chunks no longer than max_days."""
    if max_days < 1:
        raise ValueError("max_days must be positive")
    current = pd.Timestamp(start)
    end = pd.Timestamp(end)
    while current <= end:
        chunk_end = min(end, current + pd.Timedelta(days=max_days - 1))
        yield current, chunk_end
        current = chunk_end + pd.Timedelta(days=1)


def json_safe(value):
    """Convert NumPy/Pandas scalars and non-finite values into strict JSON data."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        numeric = float(value)
        return numeric if np.isfinite(numeric) else None
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if value is pd.NA or value is pd.NaT:
        return None
    return value


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_parquet(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    tmp.replace(path)
    return path

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(a))

def nearest_points(
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    left_lat: str = "latitude",
    left_lon: str = "longitude",
    right_lat: str = "latitude",
    right_lon: str = "longitude",
    right_id: str = "station_id",
    prefix: str = "nearest",
) -> pd.DataFrame:
    """Attach the closest right-side point to every valid left-side point."""
    out = left.copy()
    valid_left = out[[left_lat, left_lon]].notna().all(axis=1)
    valid_right = right[[right_lat, right_lon]].notna().all(axis=1)
    candidates = right.loc[valid_right].reset_index(drop=True)
    if not valid_left.any():
        out[f"{prefix}_id"] = pd.NA
        out[f"{prefix}_distance_km"] = np.nan
        return out
    if candidates.empty:
        out[f"{prefix}_id"] = pd.NA
        out[f"{prefix}_distance_km"] = np.nan
        return out

    tree = BallTree(
        np.radians(candidates[[right_lat, right_lon]].to_numpy(dtype=float)),
        metric="haversine",
    )
    distance, index = tree.query(
        np.radians(out.loc[valid_left, [left_lat, left_lon]].to_numpy(dtype=float)),
        k=1,
    )
    matched = candidates.iloc[index[:, 0]]
    out.loc[valid_left, f"{prefix}_id"] = matched[right_id].astype(str).to_numpy()
    out.loc[valid_left, f"{prefix}_distance_km"] = distance[:, 0] * 6371.0088

    for col in right.columns:
        if col in {right_id, right_lat, right_lon}:
            continue
        out.loc[valid_left, f"{prefix}_{col}"] = matched[col].to_numpy()
    out.loc[valid_left, f"{prefix}_latitude"] = matched[right_lat].to_numpy()
    out.loc[valid_left, f"{prefix}_longitude"] = matched[right_lon].to_numpy()
    return out

def parse_damage(value: Any) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return 0.0
    text = str(value).strip().upper().replace(",", "")
    if not text:
        return 0.0
    match = re.fullmatch(r"([0-9]*\.?[0-9]+)\s*([KMB]?)", text)
    if not match:
        return 0.0
    number = float(match.group(1))
    return number * {"": 1.0, "K": 1e3, "M": 1e6, "B": 1e9}[match.group(2)]

def rescue_count_from_text(text: Any) -> float:
    """Weak-label rescue counts from narratives; local verified logs are preferred."""
    text = str(text or "")
    patterns = [
        r"(\d{1,4})\s+(?:people|persons|swimmers|surfers|victims|children|teens?)\s+(?:were\s+)?rescued",
        r"rescued\s+(\d{1,4})\s+(?:people|persons|swimmers|surfers|victims|children|teens?)",
        r"(\d{1,4})\s+(?:water\s+)?rescues?",
    ]
    values = []
    for pattern in patterns:
        values.extend(int(x) for x in re.findall(pattern, text, flags=re.I))
    return float(max(values)) if values else np.nan

## 5. Collect NOAA Storm Events hazard reports

In [ ]:
STORM_DIR = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
STORM_DETAILS_RE = re.compile(
    r"(StormEvents_details-ftp_v1\.0_d(?P<year>\d{4})_c(?P<created>\d{8})\.csv\.gz)"
)

def discover_storm_detail_files() -> dict[int, str]:
    html = get_bytes(STORM_DIR).decode("utf-8", errors="replace")
    found: dict[int, tuple[str, str]] = {}
    for filename, year_text, created in STORM_DETAILS_RE.findall(html):
        year = int(year_text)
        if HISTORICAL_START_YEAR <= year <= HISTORICAL_END_YEAR:
            if year not in found or created > found[year][0]:
                found[year] = (created, filename)
    if not found:
        raise RuntimeError("No NOAA Storm Events details files discovered.")
    return {year: urljoin(STORM_DIR, filename) for year, (_, filename) in sorted(found.items())}

def normalize_storm_events(frame: pd.DataFrame) -> pd.DataFrame:
    frame.columns = [str(c).strip().upper() for c in frame.columns]
    for required in ("STATE", "EVENT_TYPE", "BEGIN_DATE_TIME"):
        if required not in frame.columns:
            frame[required] = pd.NA
    frame["STATE"] = frame["STATE"].astype(str).str.upper().str.strip()
    frame["STATE_ABBR"] = frame["STATE"].map(STATE_NAME_TO_ABBR)
    frame["EVENT_TYPE"] = frame["EVENT_TYPE"].astype(str).str.strip()

    keep = frame["STATE_ABBR"].isin(COASTAL_STATES) & frame["EVENT_TYPE"].isin(DANGEROUS_EVENT_TYPES)
    frame = frame.loc[keep].copy()

    start = pd.to_datetime(frame.get("BEGIN_DATE_TIME"), errors="coerce", utc=True)
    end = pd.to_datetime(frame.get("END_DATE_TIME"), errors="coerce", utc=True)
    frame["event_time"] = start
    frame["end_time"] = end.fillna(start + pd.Timedelta(hours=1))

    for col in ["BEGIN_LAT", "BEGIN_LON", "END_LAT", "END_LON",
                "DEATHS_DIRECT", "DEATHS_INDIRECT", "INJURIES_DIRECT", "INJURIES_INDIRECT"]:
        frame[col] = pd.to_numeric(frame.get(col), errors="coerce")

    frame["latitude"] = frame["BEGIN_LAT"].fillna(frame["END_LAT"])
    frame["longitude"] = frame["BEGIN_LON"].fillna(frame["END_LON"])
    frame["deaths"] = frame[["DEATHS_DIRECT", "DEATHS_INDIRECT"]].fillna(0).sum(axis=1)
    frame["injuries"] = frame[["INJURIES_DIRECT", "INJURIES_INDIRECT"]].fillna(0).sum(axis=1)
    frame["damage_usd"] = (
        frame.get("DAMAGE_PROPERTY", pd.Series(index=frame.index, dtype=object)).map(parse_damage)
        + frame.get("DAMAGE_CROPS", pd.Series(index=frame.index, dtype=object)).map(parse_damage)
    )

    narrative = (
        frame.get("EVENT_NARRATIVE", pd.Series("", index=frame.index)).fillna("").astype(str)
        + " "
        + frame.get("EPISODE_NARRATIVE", pd.Series("", index=frame.index)).fillna("").astype(str)
    )
    frame["narrative"] = narrative.str.strip()
    frame["rescued_count_weak"] = frame["narrative"].map(rescue_count_from_text)

    # Severity is intentionally transparent and bounded.
    frame["severity"] = (
        1.0
        + 5.0 * frame["deaths"].clip(0, 10)
        + 2.0 * frame["injuries"].clip(0, 25)
        + np.log1p(frame["damage_usd"].clip(lower=0)) / 10.0
        + np.log1p(frame["rescued_count_weak"].fillna(0)) / 2.0
    ).clip(1.0, 25.0)

    frame["event_id"] = (
        frame.get("EVENT_ID", pd.Series(index=frame.index, dtype=object))
        .astype("string")
        .fillna(
            frame["STATE_ABBR"].astype(str)
            + "-"
            + frame["event_time"].astype(str)
            + "-"
            + frame["EVENT_TYPE"].astype(str)
        )
    )
    frame["event_type"] = frame["EVENT_TYPE"]
    frame["state"] = frame["STATE_ABBR"]
    frame["location_name"] = (
        frame.get("CZ_NAME", pd.Series("", index=frame.index)).fillna("").astype(str)
    )
    frame["source"] = "NOAA Storm Events"
    frame["verified"] = True

    columns = [
        "event_id", "event_time", "end_time", "event_type", "state",
        "location_name", "latitude", "longitude", "deaths", "injuries",
        "damage_usd", "rescued_count_weak", "severity", "narrative",
        "source", "verified",
    ]
    return frame[columns].dropna(subset=["event_time"]).reset_index(drop=True)

def collect_storm_events(force: bool = False) -> pd.DataFrame:
    output = PROCESSED / "storm_events_contiguous_coasts.parquet"
    if output.exists() and not force:
        return pd.read_parquet(output)

    files = discover_storm_detail_files()
    frames = []
    years = list(files)
    if not FULL_SCALE:
        # Keep all years but only the already filtered event rows, so demo collection stays modest.
        pass

    from tqdm.auto import tqdm
    for year in tqdm(years, desc="NOAA Storm Events years"):
        raw_path = RAW / "storm_events" / f"details_{year}.csv.gz"
        raw_path.parent.mkdir(parents=True, exist_ok=True)
        if not raw_path.exists():
            raw_path.write_bytes(get_bytes(files[year]))
        with gzip.open(raw_path, "rb") as f:
            frame = pd.read_csv(f, low_memory=False)
        normalized = normalize_storm_events(frame)
        if not normalized.empty:
            frames.append(normalized)

    if not frames:
        raise RuntimeError("NOAA Storm Events returned no matching coastal hazard rows.")
    events = pd.concat(frames, ignore_index=True)
    atomic_parquet(events, output)
    return events

events = collect_storm_events()
events = require_frame(
    events,
    "NOAA Storm Events coastal hazard collection",
    ["event_id", "event_time", "event_type", "state", "latitude", "longitude"],
)
print("Historical coastal hazard rows:", len(events))
display(events.head())
display(events.groupby(["state", "event_type"]).size().rename("events").reset_index().head(30))

## 6. Add verified lifeguard, fire-rescue, or local beach reports

In [ ]:
CUSTOM_COLUMNS = [
    "event_time", "end_time", "beach_name", "state", "latitude", "longitude",
    "rescued_count", "injuries", "deaths", "source_url", "verified", "narrative",
]

def load_custom_rescue_reports(path: str | Path | None) -> pd.DataFrame:
    if not path:
        return pd.DataFrame()
    path = Path(path)
    custom = pd.read_csv(path)
    missing = set(CUSTOM_COLUMNS) - set(custom.columns)
    if missing:
        raise ValueError(f"Custom report CSV is missing columns: {sorted(missing)}")

    custom["event_time"] = pd.to_datetime(custom["event_time"], errors="coerce", utc=True)
    custom["end_time"] = pd.to_datetime(custom["end_time"], errors="coerce", utc=True)
    custom["end_time"] = custom["end_time"].fillna(custom["event_time"] + pd.Timedelta(hours=1))
    custom["state"] = custom["state"].astype(str).str.upper().str.strip()
    custom = custom[custom["state"].isin(COASTAL_STATES)].copy()
    verified_text = custom["verified"].astype(str).str.strip().str.lower()
    custom["verified"] = verified_text.isin({"1", "true", "yes", "y"})
    custom = custom[custom["verified"]].copy()

    for c in ["latitude", "longitude"]:
        custom[c] = pd.to_numeric(custom[c], errors="coerce")
    custom = custom.dropna(subset=["event_time", "latitude", "longitude"])
    for c in ["rescued_count", "injuries", "deaths"]:
        custom[c] = pd.to_numeric(custom[c], errors="coerce").fillna(0)

    custom["event_id"] = [
        hashlib.sha1(f"custom|{t}|{lat}|{lon}|{i}".encode()).hexdigest()[:20]
        for i, (t, lat, lon) in enumerate(
            zip(custom["event_time"], custom["latitude"], custom["longitude"])
        )
    ]
    custom["event_type"] = "Verified Rescue Report"
    custom["location_name"] = custom["beach_name"]
    custom["damage_usd"] = 0.0
    custom["rescued_count_weak"] = custom["rescued_count"]
    custom["severity"] = (
        1
        + np.log1p(custom["rescued_count"]) / 2
        + 2 * custom["injuries"]
        + 5 * custom["deaths"]
    ).clip(1, 25)
    custom["source"] = custom["source_url"].fillna("Local verified report")
    return custom[[
        "event_id", "event_time", "end_time", "event_type", "state",
        "location_name", "latitude", "longitude", "deaths", "injuries",
        "damage_usd", "rescued_count_weak", "severity", "narrative",
        "source", "verified",
    ]]

custom_events = load_custom_rescue_reports(CUSTOM_RESCUE_REPORTS_CSV)
if not custom_events.empty:
    events = pd.concat([events, custom_events], ignore_index=True)
    events = events.drop_duplicates("event_id")
    atomic_parquet(events, PROCESSED / "events_with_custom_rescues.parquet")

print({"official_events": len(events) - len(custom_events), "custom_verified_reports": len(custom_events)})

## 7. Download the USGS GNIS beach catalog

In [ ]:
GNIS_QUERY = "https://carto.nationalmap.gov/arcgis/rest/services/geonames/MapServer/5/query"

def _first_coordinate(geometry: dict | None) -> tuple[float, float]:
    if not geometry:
        return (np.nan, np.nan)
    coords = geometry.get("coordinates")
    if coords is None:
        return (np.nan, np.nan)
    # Landforms is commonly returned as MultiPoint.
    while isinstance(coords, list) and coords and isinstance(coords[0], list):
        coords = coords[0]
    if isinstance(coords, list) and len(coords) >= 2:
        return (float(coords[1]), float(coords[0]))
    return (np.nan, np.nan)

def collect_gnis_beaches(force: bool = False) -> pd.DataFrame:
    output = PROCESSED / "gnis_beaches_contiguous_ocean_states.parquet"
    if output.exists() and not force:
        return pd.read_parquet(output)

    state_sql = ",".join(f"'{s}'" for s in COASTAL_STATES)
    where = f"gaz_featureclass = 'Beach' AND state_alpha IN ({state_sql})"
    offset = 0
    page_size = 2000
    records = []

    while True:
        payload = get_json(
            GNIS_QUERY,
            params={
                "where": where,
                "outFields": "gaz_id,gaz_name,gaz_featureclass,state_alpha,county_name",
                "returnGeometry": "true",
                "outSR": "4326",
                "f": "geojson",
                "resultOffset": offset,
                "resultRecordCount": page_size,
                "orderByFields": "gaz_id",
            },
        )
        features = payload.get("features", [])
        for feature in features:
            props = feature.get("properties", {})
            lat, lon = _first_coordinate(feature.get("geometry"))
            records.append({
                "beach_id": str(props.get("gaz_id")),
                "beach_name": props.get("gaz_name"),
                "state": props.get("state_alpha"),
                "county": props.get("county_name"),
                "latitude": lat,
                "longitude": lon,
                "source": "USGS GNIS",
            })
        if len(features) < page_size:
            break
        offset += page_size
        time.sleep(0.25)

    beaches = pd.DataFrame(records, columns=[
        "beach_id", "beach_name", "state", "county", "latitude", "longitude", "source"
    ])
    beaches = beaches[
        beaches["state"].isin(COASTAL_STATES)
        & beaches[["latitude", "longitude"]].notna().all(axis=1)
    ].drop_duplicates("beach_id")
    atomic_parquet(beaches, output)
    return beaches

beaches = collect_gnis_beaches()
beaches = require_frame(
    beaches,
    "USGS GNIS beach catalog",
    ["beach_id", "beach_name", "state", "latitude", "longitude"],
)
print("Named beaches:", len(beaches))
display(beaches.sample(min(10, len(beaches)), random_state=RANDOM_SEED))

## 8. NOAA CO-OPS tide stations and NOAA NDBC buoy stations

In [ ]:
COOPS_MDAPI = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations.json"
NDBC_ACTIVE_XML = "https://www.ndbc.noaa.gov/activestations.xml"

def collect_coops_stations(force: bool = False) -> pd.DataFrame:
    output = PROCESSED / "coops_tide_prediction_stations.parquet"
    if output.exists() and not force:
        return pd.read_parquet(output)

    payload = get_json(COOPS_MDAPI, params={"type": "tidepredictions", "units": "metric"})
    records = []
    for station in payload.get("stations", []):
        lat = pd.to_numeric(station.get("lat"), errors="coerce")
        lon = pd.to_numeric(station.get("lng"), errors="coerce")
        state = str(station.get("state") or "").upper().strip()
        records.append({
            "station_id": str(station.get("id")),
            "station_name": station.get("name"),
            "state": state,
            "latitude": lat,
            "longitude": lon,
            "tidal": station.get("tidal"),
            "timezone": station.get("timezone"),
            "source": "NOAA CO-OPS",
        })
    stations = pd.DataFrame(records, columns=[
        "station_id", "station_name", "state", "latitude", "longitude",
        "tidal", "timezone", "source"
    ])
    # Some metadata rows omit state. Geographic bounds retain ocean-facing CONUS candidates.
    geographic = (
        stations["latitude"].between(24.0, 50.5)
        & stations["longitude"].between(-130.0, -65.0)
    )
    stations = stations[(stations["state"].isin(COASTAL_STATES)) | (stations["state"].eq("") & geographic)]
    stations = stations.dropna(subset=["latitude", "longitude"]).drop_duplicates("station_id")
    atomic_parquet(stations, output)
    return stations

def collect_ndbc_active_stations(force: bool = False) -> pd.DataFrame:
    output = PROCESSED / "ndbc_active_stations.parquet"
    if output.exists() and not force:
        return pd.read_parquet(output)

    import xml.etree.ElementTree as ET
    root = ET.fromstring(get_bytes(NDBC_ACTIVE_XML))
    rows = []
    for node in root.findall(".//station"):
        row = node.attrib.copy()
        rows.append({
            "buoy_id": str(row.get("id", "")).upper(),
            "buoy_name": row.get("name"),
            "owner": row.get("owner"),
            "program": row.get("pgm"),
            "station_type": row.get("type"),
            "latitude": pd.to_numeric(row.get("lat"), errors="coerce"),
            "longitude": pd.to_numeric(row.get("lon"), errors="coerce"),
            "met": row.get("met"),
            "currents": row.get("currents"),
            "source": "NOAA NDBC",
        })
    stations = pd.DataFrame(rows, columns=[
        "buoy_id", "buoy_name", "owner", "program", "station_type",
        "latitude", "longitude", "met", "currents", "source"
    ])
    stations = stations[
        stations["latitude"].between(20.0, 52.0)
        & stations["longitude"].between(-135.0, -60.0)
    ].dropna(subset=["latitude", "longitude"]).drop_duplicates("buoy_id")
    atomic_parquet(stations, output)
    return stations

coops = collect_coops_stations()
coops = require_frame(
    coops,
    "NOAA CO-OPS tide-prediction station catalog",
    ["station_id", "latitude", "longitude"],
)
ndbc = collect_ndbc_active_stations()
if ndbc.empty:
    print("Warning: NDBC active-station catalog is empty; tide-only training will continue.")

beaches = nearest_points(beaches, coops, right_id="station_id", prefix="tide")
beaches = beaches[beaches["tide_distance_km"] <= MAX_BEACH_TO_STATION_KM].copy()
require_frame(beaches, "beaches mapped to CO-OPS stations", ["beach_id", "tide_id"])

events = nearest_points(events, coops, right_id="station_id", prefix="tide")
events = events[events["tide_distance_km"] <= MAX_EVENT_TO_STATION_KM].copy()
require_frame(events, "events mapped to CO-OPS stations", ["tide_id", "event_time"])

coops_with_buoy = nearest_points(coops, ndbc, right_id="buoy_id", prefix="buoy")
coops_with_buoy.loc[
    coops_with_buoy["buoy_distance_km"] > MAX_STATION_TO_BUOY_KM,
    "buoy_id",
] = pd.NA

atomic_parquet(beaches, PROCESSED / "beaches_with_tide_station.parquet")
atomic_parquet(events, PROCESSED / "events_with_tide_station.parquet")
atomic_parquet(coops_with_buoy, PROCESSED / "coops_with_nearest_buoy.parquet")

print({
    "coops_stations": len(coops),
    "active_ndbc_stations": len(ndbc),
    "mapped_beaches": len(beaches),
    "mapped_events": len(events),
})
display(beaches.head())

## 9. Select the historical station-year workload

In [ ]:
events["year"] = events["event_time"].dt.year
station_year_counts = (
    events.dropna(subset=["tide_id", "year"])
    .groupby(["tide_id", "year"])
    .size()
    .rename("event_count")
    .reset_index()
    .sort_values(["event_count", "year"], ascending=[False, False])
)

if station_year_counts.empty:
    raise RuntimeError(
        "No event-associated tide-station years were found. Increase the station distance, "
        "inspect event coordinates, or verify the CO-OPS station catalog."
    )

max_groups = FULL_MAX_STATION_YEARS if FULL_SCALE else DEMO_MAX_STATION_YEARS
if max_groups is not None:
    station_year_work = station_year_counts.head(max_groups).copy()
else:
    station_year_work = station_year_counts.copy()

# Add a small amount of context years around each event year.
expanded = []
for row in station_year_work.itertuples(index=False):
    for y in sorted({int(row.year) - 1, int(row.year), int(row.year) + 1}):
        if HISTORICAL_START_YEAR <= y <= HISTORICAL_END_YEAR:
            expanded.append({"station_id": str(row.tide_id), "year": y, "event_count": row.event_count})
station_year_work = (
    pd.DataFrame(expanded, columns=["station_id", "year", "event_count"])
    .drop_duplicates(["station_id", "year"])
    .sort_values(["event_count", "year"], ascending=[False, False])
    .reset_index(drop=True)
)

station_year_work.to_csv(REPORTS / "station_year_workload.csv", index=False)
print("Station-year partitions:", len(station_year_work))
display(station_year_work.head(20))

## 10. Collect historical CO-OPS tides

In [ ]:
COOPS_DATA_API = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"


def _coops_data_single(
    station_id: str,
    begin_date: str,
    end_date: str,
    product: str,
    *,
    datum: str = "MLLW",
    interval: str | None = None,
) -> pd.DataFrame:
    params = {
        "product": product,
        "application": "SurfGuard-USA",
        "begin_date": begin_date,
        "end_date": end_date,
        "station": station_id,
        "time_zone": "gmt",
        "units": "metric",
        "format": "json",
    }
    if product not in {"wind", "air_temperature", "water_temperature", "air_pressure"}:
        params["datum"] = datum
    if interval:
        params["interval"] = interval

    payload = get_json(COOPS_DATA_API, params=params, timeout=180)
    rows = payload.get("predictions") or payload.get("data") or []
    if not rows:
        return pd.DataFrame(columns=["time", "value"])
    frame = pd.DataFrame(rows)
    if "t" not in frame.columns or "v" not in frame.columns:
        raise ValueError(f"Unexpected CO-OPS response columns: {frame.columns.tolist()}")
    frame["time"] = pd.to_datetime(frame["t"], errors="coerce", utc=True)
    frame["value"] = pd.to_numeric(frame["v"], errors="coerce")
    return frame[["time", "value"]].dropna(subset=["time"]).sort_values("time")


def coops_data(
    station_id: str,
    begin_date: str,
    end_date: str,
    product: str,
    *,
    datum: str = "MLLW",
    interval: str | None = None,
) -> pd.DataFrame:
    """Retrieve CO-OPS data while respecting the service's 31-day request cap."""
    start = pd.Timestamp(begin_date)
    end = pd.Timestamp(end_date)
    if end < start:
        raise ValueError("end_date precedes begin_date")

    frames = []
    for chunk_start, chunk_end in date_chunks(start, end, max_days=COOPS_MAX_REQUEST_DAYS):
        frames.append(
            _coops_data_single(
                station_id,
                chunk_start.strftime("%Y%m%d"),
                chunk_end.strftime("%Y%m%d"),
                product,
                datum=datum,
                interval=interval,
            )
        )
        if chunk_end < end:
            time.sleep(0.08)

    populated = [frame for frame in frames if not frame.empty]
    if not populated:
        return pd.DataFrame(columns=["time", "value"])
    return (
        pd.concat(populated, ignore_index=True)
        .drop_duplicates("time")
        .sort_values("time")
        .reset_index(drop=True)
    )


def collect_tide_station_year(station_id: str, year: int, force: bool = False) -> Path:
    output = RAW / "coops" / f"station={station_id}" / f"year={year}" / "tides.parquet"
    if output.exists() and not force:
        return output

    begin = f"{year}0101"
    end = f"{year}1231"
    predictions = coops_data(station_id, begin, end, "predictions", interval="h")
    predictions = predictions.rename(columns={"value": "tide_pred_m"})
    if predictions.empty:
        raise RuntimeError(f"No hourly tide predictions returned for station {station_id} in {year}")

    try:
        observed = coops_data(station_id, begin, end, "hourly_height")
        observed = observed.rename(columns={"value": "tide_obs_m"})
    except Exception as exc:
        print(f"Observed tide unavailable {station_id} {year}: {type(exc).__name__}")
        observed = pd.DataFrame(columns=["time", "tide_obs_m"])

    tide = predictions.merge(observed, on="time", how="outer").sort_values("time")
    tide["station_id"] = str(station_id)
    tide["year"] = int(year)
    tide["surge_residual_m"] = tide["tide_obs_m"] - tide["tide_pred_m"]

    for base in ["tide_pred_m", "tide_obs_m", "surge_residual_m"]:
        tide[f"{base}_slope_1h"] = tide[base].diff(1)
        tide[f"{base}_slope_3h"] = tide[base].diff(3)

    tide["tide_percentile_station_year"] = tide["tide_pred_m"].rank(pct=True)
    atomic_parquet(tide, output)
    return output


from tqdm.auto import tqdm

tide_paths = []
for row in tqdm(station_year_work.itertuples(index=False), total=len(station_year_work), desc="CO-OPS station-years"):
    try:
        tide_paths.append(collect_tide_station_year(str(row.station_id), int(row.year)))
    except Exception as exc:
        print("CO-OPS partition failed:", row.station_id, row.year, type(exc).__name__, str(exc)[:180])
    time.sleep(0.15)

if not tide_paths:
    raise RuntimeError("No CO-OPS tide partitions were collected; cannot build the training panel.")
print("Collected tide partitions:", len(tide_paths))

## 11. Collect historical NDBC buoy weather and wave observations

In [ ]:
NDBC_STDMET = "https://www.ndbc.noaa.gov/data/historical/stdmet"
_NDBC_INDEX_CACHE: dict[tuple[str, int], str | None] = {}


def discover_ndbc_stdmet_url(station_id: str, year: int) -> str | None:
    key = (station_id.lower(), int(year))
    if key in _NDBC_INDEX_CACHE:
        return _NDBC_INDEX_CACHE[key]
    html = get_bytes(NDBC_STDMET + "/", timeout=180).decode("utf-8", errors="replace")
    target = f"{station_id.lower()}h{year}.txt.gz"
    soup = BeautifulSoup(html, "html.parser")
    for anchor in soup.find_all("a", href=True):
        filename = Path(anchor["href"]).name
        if filename.lower() == target:
            url = urljoin(NDBC_STDMET + "/", filename)
            _NDBC_INDEX_CACHE[key] = url
            return url
    _NDBC_INDEX_CACHE[key] = None
    return None


def parse_ndbc_stdmet(content: bytes, station_id: str) -> pd.DataFrame:
    text = gzip.decompress(content).decode("utf-8", errors="replace")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return pd.DataFrame()

    raw_header = lines[0].lstrip("#").split()
    # NDBC uses MM for month and lowercase mm for minute. Blind uppercasing
    # creates duplicate MM columns, so normalize the minute field explicitly.
    header = []
    for token in raw_header:
        clean = str(token).lstrip("#")
        if clean == "mm":
            header.append("MN")
        elif clean.lower() == "hh":
            header.append("HH")
        else:
            header.append(clean.upper())
    if len(header) != len(set(header)):
        raise ValueError(f"Duplicate NDBC columns after normalization: {header}")
    data_lines = [line for line in lines[1:] if not line.startswith("#")]
    frame = pd.read_csv(io.StringIO("\n".join(data_lines)), sep=r"\s+", names=header, engine="python")

    year_col = "YYYY" if "YYYY" in frame.columns else "YY"
    required = [year_col, "MM", "DD", "HH"]
    if not all(c in frame.columns for c in required):
        raise ValueError(f"Unexpected NDBC header: {frame.columns.tolist()[:20]}")

    for column in frame.columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    years = frame[year_col]
    if year_col == "YY":
        years = np.where(years < 70, 2000 + years, 1900 + years)

    minute = frame["MN"] if "MN" in frame.columns else 0
    frame["time"] = pd.to_datetime(
        {
            "year": years,
            "month": frame["MM"],
            "day": frame["DD"],
            "hour": frame["HH"],
            "minute": minute,
        },
        errors="coerce",
        utc=True,
    )

    feature_cols = ["WDIR", "WSPD", "GST", "WVHT", "DPD", "APD", "MWD", "PRES", "ATMP", "WTMP", "DEWP", "VIS", "TIDE"]
    missing_sentinels = {
        "WDIR": 999.0,
        "WSPD": 99.0,
        "GST": 99.0,
        "WVHT": 99.0,
        "DPD": 99.0,
        "APD": 99.0,
        "MWD": 999.0,
        "PRES": 9999.0,
        "ATMP": 999.0,
        "WTMP": 999.0,
        "DEWP": 999.0,
        "VIS": 99.0,
        "TIDE": 99.0,
    }
    for column in feature_cols:
        if column not in frame:
            frame[column] = np.nan
        sentinel = missing_sentinels[column]
        frame.loc[np.isclose(frame[column].abs(), sentinel) | (frame[column].abs() > sentinel), column] = np.nan

    output = frame[["time", *feature_cols]].copy()
    output["buoy_id"] = station_id.upper()
    output["wave_energy_proxy"] = output["WVHT"].pow(2) * output["DPD"]
    output["wave_steepness_proxy"] = output["WVHT"] / np.maximum(output["DPD"].pow(2), 0.25)
    return output.dropna(subset=["time"]).sort_values("time")


def collect_ndbc_station_year(station_id: str, year: int, force: bool = False) -> Path | None:
    output = RAW / "ndbc" / f"station={station_id.upper()}" / f"year={year}" / "stdmet.parquet"
    if output.exists() and not force:
        return output
    url = discover_ndbc_stdmet_url(station_id, year)
    if not url:
        return None
    try:
        content = get_bytes(url, timeout=180)
    except Exception:
        return None
    frame = parse_ndbc_stdmet(content, station_id)
    if frame.empty:
        return None
    atomic_parquet(frame, output)
    return output


work_with_buoy = station_year_work.merge(
    coops_with_buoy[["station_id", "buoy_id", "buoy_distance_km"]],
    on="station_id",
    how="left",
)
buoy_work = work_with_buoy.dropna(subset=["buoy_id"])[["buoy_id", "year"]].drop_duplicates()

buoy_paths = []
for row in tqdm(buoy_work.itertuples(index=False), total=len(buoy_work), desc="NDBC station-years"):
    path = collect_ndbc_station_year(str(row.buoy_id), int(row.year))
    if path:
        buoy_paths.append(path)
    time.sleep(0.1)

print("Collected buoy partitions:", len(buoy_paths))

## 12. Build the hourly feature panel

In [ ]:
def read_partition(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)

def build_hourly_panel(force: bool = False) -> pd.DataFrame:
    output = PROCESSED / "hourly_ocean_feature_panel.parquet"
    if output.exists() and not force:
        return pd.read_parquet(output)

    station_meta = coops_with_buoy.set_index("station_id")
    frames = []
    for path in tqdm(tide_paths, desc="Joining tide + buoy partitions"):
        tide = read_partition(path)
        station_id = str(tide["station_id"].iloc[0])
        year = int(tide["year"].iloc[0])
        tide = tide.sort_values("time")

        if station_id in station_meta.index:
            meta = station_meta.loc[station_id]
            tide["station_name"] = meta["station_name"]
            tide["station_latitude"] = meta["latitude"]
            tide["station_longitude"] = meta["longitude"]
            tide["station_state"] = meta["state"]
            buoy_id = meta.get("buoy_id")
        else:
            buoy_id = None

        buoy_path = (
            RAW / "ndbc" / f"station={str(buoy_id).upper()}" / f"year={year}" / "stdmet.parquet"
            if pd.notna(buoy_id)
            else None
        )
        if buoy_path and buoy_path.exists():
            buoy = pd.read_parquet(buoy_path).sort_values("time")
            tide = pd.merge_asof(
                tide.sort_values("time"),
                buoy.sort_values("time"),
                on="time",
                direction="nearest",
                tolerance=pd.Timedelta(minutes=90),
            )
        frames.append(tide)

    if not frames:
        raise RuntimeError("No tide partitions were available to build the hourly feature panel.")
    panel = pd.concat(frames, ignore_index=True).sort_values(["station_id", "time"])
    panel["month"] = panel["time"].dt.month
    panel["dayofyear"] = panel["time"].dt.dayofyear
    panel["hour"] = panel["time"].dt.hour
    panel["weekday"] = panel["time"].dt.weekday
    panel["is_weekend"] = (panel["weekday"] >= 5).astype(int)

    # Smooth cyclical time encodings.
    panel["hour_sin"] = np.sin(2 * np.pi * panel["hour"] / 24)
    panel["hour_cos"] = np.cos(2 * np.pi * panel["hour"] / 24)
    panel["doy_sin"] = np.sin(2 * np.pi * panel["dayofyear"] / 365.25)
    panel["doy_cos"] = np.cos(2 * np.pi * panel["dayofyear"] / 365.25)

    # Beach attendance proxies: weekends, daylight-like hours, and warm season.
    panel["daylight_proxy"] = panel["hour"].between(7, 19).astype(int)
    panel["warm_season_proxy"] = panel["month"].between(5, 9).astype(int)
    panel["crowd_proxy"] = (
        0.4 * panel["is_weekend"]
        + 0.35 * panel["daylight_proxy"]
        + 0.25 * panel["warm_season_proxy"]
    )

    atomic_parquet(panel, output)
    return panel

panel = build_hourly_panel()
print("Hourly feature rows:", f"{len(panel):,}")
display(panel.head())

## 13. Label historical dangerous windows without leaking narratives into features

In [ ]:
def apply_event_labels(panel: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    labeled = panel.copy()
    labeled["danger_label"] = np.uint8(0)
    labeled["event_severity"] = np.float32(0)
    labeled["rescued_count_target"] = np.float32(0)

    event_subset = events.dropna(subset=["tide_id", "event_time"]).copy()
    event_subset["tide_id"] = event_subset["tide_id"].astype(str)

    for station_id, idx in labeled.groupby("station_id", sort=False).groups.items():
        idx = np.asarray(list(idx))
        order = np.argsort(labeled.loc[idx, "time"].to_numpy())
        sorted_idx = idx[order]
        times = labeled.loc[sorted_idx, "time"].astype("int64").to_numpy()

        station_events = event_subset[event_subset["tide_id"] == str(station_id)]
        if station_events.empty:
            continue

        labels = np.zeros(len(sorted_idx), dtype=np.uint8)
        severity = np.zeros(len(sorted_idx), dtype=np.float32)
        rescue = np.zeros(len(sorted_idx), dtype=np.float32)

        for event in station_events.itertuples(index=False):
            start = pd.Timestamp(event.event_time) - pd.Timedelta(hours=LABEL_WINDOW_HOURS)
            end = pd.Timestamp(event.end_time) + pd.Timedelta(hours=LABEL_WINDOW_HOURS)
            left = np.searchsorted(times, start.value, side="left")
            right = np.searchsorted(times, end.value, side="right")
            if right <= left:
                continue
            labels[left:right] = 1
            severity[left:right] = np.maximum(severity[left:right], float(event.severity))
            rescue_value = getattr(event, "rescued_count_weak", np.nan)
            if pd.notna(rescue_value):
                rescue[left:right] = np.maximum(rescue[left:right], float(rescue_value))

        labeled.loc[sorted_idx, "danger_label"] = labels
        labeled.loc[sorted_idx, "event_severity"] = severity
        labeled.loc[sorted_idx, "rescued_count_target"] = rescue

    return labeled

labeled_panel = apply_event_labels(panel, events)
atomic_parquet(labeled_panel, PROCESSED / "hourly_labeled_panel.parquet")

print(labeled_panel["danger_label"].value_counts(dropna=False))
print("Positive rate:", labeled_panel["danger_label"].mean())

## 14. Build a balanced training table

In [ ]:
FEATURE_COLUMNS = [
    "tide_pred_m",
    "tide_obs_m",
    "surge_residual_m",
    "tide_pred_m_slope_1h",
    "tide_pred_m_slope_3h",
    "tide_obs_m_slope_1h",
    "tide_obs_m_slope_3h",
    "surge_residual_m_slope_1h",
    "surge_residual_m_slope_3h",
    "tide_percentile_station_year",
    "WVHT",
    "DPD",
    "APD",
    "MWD",
    "WSPD",
    "GST",
    "PRES",
    "ATMP",
    "WTMP",
    "wave_energy_proxy",
    "wave_steepness_proxy",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
    "is_weekend",
    "daylight_proxy",
    "warm_season_proxy",
    "crowd_proxy",
    "station_latitude",
    "station_longitude",
]

for c in FEATURE_COLUMNS:
    if c not in labeled_panel.columns:
        labeled_panel[c] = np.nan

positive = labeled_panel[labeled_panel["danger_label"] == 1].copy()
negative = labeled_panel[labeled_panel["danger_label"] == 0].copy()

if positive.empty:
    raise RuntimeError(
        "No positive coastal-hazard windows were created. Increase station-year coverage, "
        "verify event-to-station matching, or add verified local rescue reports before training."
    )
if negative.empty:
    raise RuntimeError("No negative/background rows were created; the classifier needs both classes.")

max_negative = min(len(negative), max(len(positive) * NEGATIVE_TO_POSITIVE_RATIO, 50_000))
if len(negative) > max_negative:
    # Balance station/month strata without DataFrameGroupBy.apply (deprecated in pandas).
    stratum_size = (
        negative.groupby(["station_id", "month"], dropna=False)["station_id"]
        .transform("size")
        .clip(lower=1)
    )
    inverse_stratum_weight = 1.0 / stratum_size.to_numpy(dtype=float)
    negative = negative.sample(
        n=max_negative,
        weights=inverse_stratum_weight,
        random_state=RANDOM_SEED,
        replace=False,
    )

dataset = (
    pd.concat([positive, negative], ignore_index=True)
    .sort_values("time")
    .reset_index(drop=True)
)

keep = ["time", "station_id", "danger_label", "event_severity", "rescued_count_target", *FEATURE_COLUMNS]
dataset = dataset[keep]
atomic_parquet(dataset, PROCESSED / "training_dataset.parquet")

print({
    "rows": len(dataset),
    "positives": int(dataset["danger_label"].sum()),
    "positive_rate": float(dataset["danger_label"].mean()),
})


## 15. Time-split, calibrated supervised ensemble

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
import joblib

class FiniteMedianImputer(BaseEstimator, TransformerMixin):
    """Fixed-width numeric imputer that never drops all-missing columns.

    Training medians are used where finite observations exist. Columns with no
    finite training values receive a deterministic fallback, and missingness
    indicators are appended without invoking NumPy nan-reduction warnings.
    """

    def __init__(self, fallback_value: float = 0.0, add_indicator: bool = True):
        self.fallback_value = float(fallback_value)
        self.add_indicator = bool(add_indicator)

    def fit(self, X, y=None):
        frame = pd.DataFrame(X).apply(pd.to_numeric, errors="coerce")
        matrix = frame.to_numpy(dtype=np.float64, copy=True)
        finite = np.isfinite(matrix)
        medians = np.full(matrix.shape[1], self.fallback_value, dtype=np.float64)
        for column in range(matrix.shape[1]):
            values = matrix[finite[:, column], column]
            if values.size:
                medians[column] = float(np.median(values))
        self.statistics_ = medians
        self.n_features_in_ = matrix.shape[1]
        self.indicator_features_ = np.flatnonzero((~finite).any(axis=0))
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(list(X.columns), dtype=object)
        return self

    def transform(self, X):
        frame = pd.DataFrame(X).apply(pd.to_numeric, errors="coerce")
        matrix = frame.to_numpy(dtype=np.float64, copy=True)
        if matrix.shape[1] != self.n_features_in_:
            raise ValueError(
                f"Expected {self.n_features_in_} features, received {matrix.shape[1]}."
            )
        missing = ~np.isfinite(matrix)
        if missing.any():
            rows, columns = np.where(missing)
            matrix[rows, columns] = self.statistics_[columns]
        if self.add_indicator and self.indicator_features_.size:
            indicators = missing[:, self.indicator_features_].astype(np.float64)
            matrix = np.hstack([matrix, indicators])
        return matrix

def _three_way_counts(n: int, train_fraction: float = 0.70, valid_fraction: float = 0.15):
    if n < 3:
        raise ValueError("At least three units are required for train/validation/test splitting.")
    n_train = max(1, int(np.floor(n * train_fraction)))
    n_valid = max(1, int(np.floor(n * valid_fraction)))
    while n_train + n_valid >= n:
        if n_train >= n_valid and n_train > 1:
            n_train -= 1
        elif n_valid > 1:
            n_valid -= 1
        else:
            break
    n_test = n - n_train - n_valid
    if n_test < 1:
        raise ValueError("Could not reserve a non-empty test split.")
    return n_train, n_valid, n_test

def event_aware_temporal_split(frame: pd.DataFrame):
    """
    Preserve chronological order inside each class and keep rows from one
    dangerous episode in the same split. This prevents validation/test from
    becoming all-negative when hazards are sparse late in the global timeline.
    """
    ordered = frame.sort_values("time").reset_index(drop=True)
    positive_rows = ordered[ordered["danger_label"].astype(int) == 1].copy()
    negative_rows = ordered[ordered["danger_label"].astype(int) == 0].copy()

    if positive_rows.empty or negative_rows.empty:
        raise RuntimeError("Training requires both positive and negative rows.")

    # Consecutive positive label windows for the same station form one incident episode.
    positive_rows = positive_rows.sort_values(["station_id", "time"]).copy()
    station_changed = positive_rows["station_id"].ne(positive_rows["station_id"].shift())
    event_gap = positive_rows["time"].diff().gt(
        pd.Timedelta(hours=max(2, 2 * LABEL_WINDOW_HOURS + 1))
    )
    positive_rows["_event_group"] = (station_changed | event_gap).cumsum()
    event_order = (
        positive_rows.groupby("_event_group", sort=False)["time"]
        .min()
        .sort_values()
        .index.to_list()
    )

    if len(event_order) >= 3:
        p_train_n, p_valid_n, _ = _three_way_counts(len(event_order))
        p_train_ids = set(event_order[:p_train_n])
        p_valid_ids = set(event_order[p_train_n:p_train_n + p_valid_n])
        p_test_ids = set(event_order[p_train_n + p_valid_n:])
        positive_parts = [
            positive_rows[positive_rows["_event_group"].isin(ids)].drop(columns="_event_group")
            for ids in (p_train_ids, p_valid_ids, p_test_ids)
        ]
        positive_split_mode = "incident-group temporal"
    else:
        # Extremely small demonstrations still get a deterministic class-preserving split.
        p_train_n, p_valid_n, _ = _three_way_counts(len(positive_rows))
        positive_parts = [
            positive_rows.iloc[:p_train_n].drop(columns="_event_group"),
            positive_rows.iloc[p_train_n:p_train_n + p_valid_n].drop(columns="_event_group"),
            positive_rows.iloc[p_train_n + p_valid_n:].drop(columns="_event_group"),
        ]
        positive_split_mode = "positive-row temporal fallback"

    negative_rows = negative_rows.sort_values("time")
    n_train_n, n_valid_n, _ = _three_way_counts(len(negative_rows))
    negative_parts = [
        negative_rows.iloc[:n_train_n],
        negative_rows.iloc[n_train_n:n_train_n + n_valid_n],
        negative_rows.iloc[n_train_n + n_valid_n:],
    ]

    splits = []
    for positive_part, negative_part in zip(positive_parts, negative_parts, strict=True):
        split = (
            pd.concat([positive_part, negative_part], ignore_index=True)
            .sort_values("time")
            .reset_index(drop=True)
        )
        if split["danger_label"].nunique() != 2:
            raise RuntimeError(
                "A model split still contains only one class. Collect more independent "
                "dangerous episodes or verified rescue reports before evaluating."
            )
        splits.append(split)

    summary = {
        "method": positive_split_mode,
        "positive_event_groups": int(len(event_order)),
        "rows": {
            "train": int(len(splits[0])),
            "validation": int(len(splits[1])),
            "test": int(len(splits[2])),
        },
        "positives": {
            "train": int(splits[0]["danger_label"].sum()),
            "validation": int(splits[1]["danger_label"].sum()),
            "test": int(splits[2]["danger_label"].sum()),
        },
    }
    return (*splits, summary)

train, valid, test, split_summary = event_aware_temporal_split(dataset)

# Remove only those requested features that contain no observed training values.
# This eliminates repeated SimpleImputer warnings while preserving every usable feature.
ALL_MISSING_FEATURE_COLUMNS = [
    c for c in FEATURE_COLUMNS if not train[c].notna().any()
]
ACTIVE_FEATURE_COLUMNS = [
    c for c in FEATURE_COLUMNS if c not in ALL_MISSING_FEATURE_COLUMNS
]
if not ACTIVE_FEATURE_COLUMNS:
    raise RuntimeError(
        "Every requested model feature is missing. Re-run tide/buoy collection and panel construction."
    )

X_train, y_train = train[ACTIVE_FEATURE_COLUMNS], train["danger_label"].astype(int)
X_valid, y_valid = valid[ACTIVE_FEATURE_COLUMNS], valid["danger_label"].astype(int)
X_test, y_test = test[ACTIVE_FEATURE_COLUMNS], test["danger_label"].astype(int)

print("Split summary:", split_summary)
print(
    f"Usable supervised features: {len(ACTIVE_FEATURE_COLUMNS)}/{len(FEATURE_COLUMNS)}"
)
if ALL_MISSING_FEATURE_COLUMNS:
    print(
        "Dropped all-missing training features (data source unavailable for this run):",
        ALL_MISSING_FEATURE_COLUMNS,
    )

base_pipeline = Pipeline([
    (
        "imputer",
        FiniteMedianImputer(
            fallback_value=0.0,
            add_indicator=True,
        ),
    ),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=350,
        max_leaf_nodes=31,
        min_samples_leaf=35,
        l2_regularization=0.8,
        early_stopping=True,
        validation_fraction=0.12,
        random_state=RANDOM_SEED,
    )),
])

positive_weight = max(1.0, (len(y_train) - y_train.sum()) / max(y_train.sum(), 1))
sample_weight = np.where(y_train.to_numpy() == 1, positive_weight, 1.0)

ensemble_seeds = [11, 23, 37] if not FULL_SCALE else [11, 23, 37, 53, 71]
ensemble = []
for seed in ensemble_seeds:
    model = clone(base_pipeline)
    model.set_params(model__random_state=seed)
    model.fit(X_train, y_train, model__sample_weight=sample_weight)
    ensemble.append(model)

def ensemble_predict(
    models,
    X: pd.DataFrame,
    feature_columns: Sequence[str] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    columns = list(feature_columns or ACTIVE_FEATURE_COLUMNS)
    safe = X.reindex(columns=columns)
    matrix = np.vstack([model.predict_proba(safe)[:, 1] for model in models])
    return matrix.mean(axis=0), matrix.std(axis=0)

valid_raw, valid_epistemic = ensemble_predict(ensemble, X_valid, ACTIVE_FEATURE_COLUMNS)
test_raw, test_epistemic = ensemble_predict(ensemble, X_test, ACTIVE_FEATURE_COLUMNS)

if y_valid.nunique() == 2 and y_valid.sum() >= 20:
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(valid_raw, y_valid)
    valid_prob = calibrator.predict(valid_raw)
    test_prob = calibrator.predict(test_raw)
else:
    calibrator = None
    valid_prob = valid_raw
    test_prob = test_raw
    print(
        "Calibration skipped: validation requires both classes and at least "
        "20 positive rows for stable isotonic calibration."
    )

def decision_cost(y_true: np.ndarray, probability: np.ndarray, threshold: float) -> float:
    prediction = probability >= threshold
    false_negative = np.sum((y_true == 1) & (~prediction))
    false_positive = np.sum((y_true == 0) & prediction)
    return 25.0 * false_negative + 1.0 * false_positive

threshold_grid = np.linspace(0.02, 0.90, 177)
alert_threshold = min(
    threshold_grid,
    key=lambda t: decision_cost(y_valid.to_numpy(), valid_prob, float(t)),
)
strong_threshold = min(0.98, max(alert_threshold + 0.15, np.quantile(valid_prob, 0.985)))

def metric_report(y_true, probability, threshold):
    y_array = np.asarray(y_true, dtype=int)
    prediction = (probability >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_array, prediction, average="binary", zero_division=0
    )
    ranking_available = len(np.unique(y_array)) == 2
    return {
        "rows": int(len(y_array)),
        "positives": int(y_array.sum()),
        "negatives": int(len(y_array) - y_array.sum()),
        "roc_auc": float(roc_auc_score(y_array, probability)) if ranking_available else None,
        "pr_auc": float(average_precision_score(y_array, probability)) if ranking_available else None,
        "brier": float(brier_score_loss(y_array, probability)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "threshold": float(threshold),
        "confusion_matrix": confusion_matrix(y_array, prediction, labels=[0, 1]).tolist(),
    }

metrics = {
    "validation": metric_report(y_valid, valid_prob, alert_threshold),
    "test": metric_report(y_test, test_prob, alert_threshold),
    "strong_threshold": float(strong_threshold),
    "positive_weight": float(positive_weight),
    "split_summary": split_summary,
    "active_features": ACTIVE_FEATURE_COLUMNS,
    "dropped_all_missing_features": ALL_MISSING_FEATURE_COLUMNS,
    "train_time_min": str(train["time"].min()),
    "train_time_max": str(train["time"].max()),
    "test_time_min": str(test["time"].min()),
    "test_time_max": str(test["time"].max()),
}

model_bundle = {
    "ensemble": ensemble,
    "calibrator": calibrator,
    "features": ACTIVE_FEATURE_COLUMNS,
    "requested_features": FEATURE_COLUMNS,
    "dropped_all_missing_features": ALL_MISSING_FEATURE_COLUMNS,
    "alert_threshold": float(alert_threshold),
    "strong_threshold": float(strong_threshold),
    "metrics": metrics,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
joblib.dump(model_bundle, MODELS / "surfguard_supervised_bundle.joblib")
(REPORTS / "supervised_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

display(
    pd.DataFrame([
        {"split": "validation", **metrics["validation"]},
        {"split": "test", **metrics["test"]},
    ]).set_index("split")
)
display(pd.Series({
    "alert_threshold": float(alert_threshold),
    "strong_threshold": float(strong_threshold),
    "positive_weight": float(positive_weight),
    "active_feature_count": len(ACTIVE_FEATURE_COLUMNS),
    "dropped_feature_count": len(ALL_MISSING_FEATURE_COLUMNS),
    "split_method": split_summary["method"],
}, name="training_summary").to_frame())


## 16. Optional rescue-demand model

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

count_rows = dataset[dataset["rescued_count_target"] > 0].copy()
rescue_demand_model = None

if len(count_rows) >= 100:
    count_train = count_rows[count_rows["time"] <= train["time"].max()]
    count_test = count_rows[count_rows["time"] > train["time"].max()]
    if len(count_train) < 50:
        print("Rescue-count rows do not overlap the training period; skipping count model.")
    else:
        rescue_demand_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", HistGradientBoostingRegressor(
                loss="poisson",
                learning_rate=0.05,
                max_iter=250,
                max_leaf_nodes=21,
                l2_regularization=1.0,
                random_state=RANDOM_SEED,
            )),
        ])
        rescue_demand_model.fit(
            count_train[FEATURE_COLUMNS],
            count_train["rescued_count_target"].clip(lower=0),
        )
        joblib.dump(rescue_demand_model, MODELS / "rescue_demand_model.joblib")
        print("Rescue-demand model trained on", len(count_train), "positive-count rows.")
else:
    print(
        "Not enough verified/weak rescue-count rows for a responsible count model.",
        "Add local lifeguard logs; incident probability remains available.",
    )

## 17. PufferForge PPO: optimize alert escalation policy

In [ ]:
# RL does not replace the supervised physical-risk estimator.
# It learns the alert action under asymmetric costs.

RL_FEATURES = [
    "ml_risk",
    "ml_uncertainty",
    "tide_percentile_station_year",
    "tide_pred_m_slope_3h",
    "surge_residual_m",
    "WVHT",
    "DPD",
    "wave_energy_proxy",
    "crowd_proxy",
    "daylight_proxy",
]

def attach_ml_scores(frame: pd.DataFrame) -> pd.DataFrame:
    scored = frame.copy()
    # Ensure all supervised feature columns exist to avoid metadata mismatches during inference
    for c in FEATURE_COLUMNS:
        if c not in scored.columns:
            scored[c] = 0.0

    raw, spread = ensemble_predict(model_bundle["ensemble"], scored[FEATURE_COLUMNS])
    probability = (
        model_bundle["calibrator"].predict(raw)
        if model_bundle["calibrator"] is not None
        else raw
    )
    scored["ml_risk"] = probability
    scored["ml_uncertainty"] = spread
    return scored

rl_frame = attach_ml_scores(dataset.copy())
for c in RL_FEATURES:
    if c not in rl_frame:
        rl_frame[c] = 0.0

# Handle columns that might be entirely NaN to suppress sklearn imputer warnings
rl_medians = rl_frame[RL_FEATURES].median(numeric_only=True).fillna(0.0)
rl_scale = rl_frame[RL_FEATURES].quantile(0.75) - rl_frame[RL_FEATURES].quantile(0.25)
rl_scale = rl_scale.replace(0, 1.0).fillna(1.0)

rl_obs = (
    (rl_frame[RL_FEATURES].fillna(rl_medians) - rl_medians) / rl_scale
).clip(-8, 8).to_numpy(dtype=np.float32)
rl_labels = rl_frame["danger_label"].to_numpy(dtype=np.int8)
rl_severity = np.maximum(1.0, rl_frame["event_severity"].to_numpy(dtype=np.float32))

class BeachAlertEnv:
    obs_size = len(RL_FEATURES)
    num_actions = 3

    def __init__(self, seed: int = 0, episode_length: int = 128):
        if len(rl_obs) < 2:
            raise RuntimeError("BeachAlertEnv requires at least two scored training rows.")
        self.rng = np.random.default_rng(seed)
        self.episode_length = min(int(episode_length), len(rl_obs) - 1)
        self.start = 0
        self.pos = 0
        self.steps = 0
        self.last_action = 0

    def reset(self, seed: int | None = None) -> np.ndarray:
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        high = max(1, len(rl_obs) - self.episode_length - 1)
        self.start = int(self.rng.integers(0, high))
        self.pos = self.start
        self.steps = 0
        self.last_action = 0
        return rl_obs[self.pos]

    def step(self, action: int):
        label = int(rl_labels[self.pos])
        severity = float(rl_severity[self.pos])
        action = int(action)

        if label:
            reward = {0: -25.0 * severity, 1: 4.0 * severity, 2: 7.0 * severity}[action]
        else:
            reward = {0: 0.15, 1: -0.55, 2: -2.25}[action]

        # Discourage rapidly oscillating public messaging.
        reward -= 0.10 * abs(action - self.last_action)
        self.last_action = action
        self.pos = min(self.pos + 1, len(rl_obs) - 1)
        self.steps += 1

        terminated = False
        truncated = self.steps >= self.episode_length or self.pos >= len(rl_obs) - 1
        return rl_obs[self.pos], float(reward), terminated, truncated, {}

    def close(self):
        pass

pufferforge_metrics = []
pufferforge_rl_available = False
pufferforge_rl_error = None
pufferforge_checkpoint_dir = MODELS / "pufferforge_alert_policy"

try:
    from pufferforge import PPOTrainer, PythonVectorEnv, TrainConfig

    num_envs = 32 if not FULL_SCALE else 128
    horizon = 64 if not FULL_SCALE else 128
    env = PythonVectorEnv(
        [lambda i=i: BeachAlertEnv(seed=RANDOM_SEED + i) for i in range(num_envs)],
        seed=RANDOM_SEED,
    )
    config = TrainConfig(
        seed=RANDOM_SEED,
        num_envs=num_envs,
        horizon=horizon,
        minibatch_size=min(1024, num_envs * horizon),
        total_timesteps=131_072 if not FULL_SCALE else 1_048_576,
        checkpoint_interval=10,
        checkpoint_dir=str(pufferforge_checkpoint_dir),
        hidden_size=128,
        hidden_layers=2,
        entropy_coef=0.015,
    )
    # Ensure exact divisibility required by PufferForge.
    while config.batch_size % config.minibatch_size != 0:
        config.minibatch_size //= 2
    config.validate()

    trainer = PPOTrainer(env, config)
    trainer.train(lambda m: pufferforge_metrics.append(m.to_dict()))
    trainer.close()
    pufferforge_rl_available = True
    pd.DataFrame(pufferforge_metrics).to_json(
        REPORTS / "pufferforge_training_metrics.jsonl",
        orient="records",
        lines=True,
    )
    print("PufferForge PPO training complete:", pufferforge_checkpoint_dir)
except Exception as exc:
    pufferforge_rl_error = f"{type(exc).__name__}: {str(exc)[:500]}"
    print("PufferForge PPO unavailable in this runtime:", pufferforge_rl_error)
    print("The calibrated supervised thresholds remain the deterministic fallback.")

print({"pufferforge_rl_available": pufferforge_rl_available, "error": pufferforge_rl_error})


## 18. Future astronomical tides for 300 days

In [ ]:
def collect_future_tides(
    station_ids: Sequence[str],
    days: int = FORECAST_DAYS,
    force: bool = False,
) -> pd.DataFrame:
    output = FORECASTS / f"future_tides_{days}d.parquet"
    if output.exists() and not force:
        return pd.read_parquet(output)

    start = pd.Timestamp.now(tz="UTC").floor("h")
    end = start + pd.Timedelta(days=days)
    frames = []

    for station_id in tqdm(sorted(set(map(str, station_ids))), desc="Future CO-OPS tides"):
        for chunk_start, chunk_end in date_chunks(start, end, max_days=COOPS_MAX_REQUEST_DAYS):
            try:
                part = coops_data(
                    station_id,
                    chunk_start.strftime("%Y%m%d"),
                    chunk_end.strftime("%Y%m%d"),
                    "predictions",
                    interval="h",
                ).rename(columns={"value": "tide_pred_m"})
                part["station_id"] = station_id
                frames.append(part)
            except Exception as exc:
                print("Future tide failed:", station_id, type(exc).__name__)
            time.sleep(0.15)

    future = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not future.empty:
        future = future.drop_duplicates(["station_id", "time"]).sort_values(["station_id", "time"])
        future["tide_pred_m_slope_1h"] = future.groupby("station_id")["tide_pred_m"].diff(1)
        future["tide_pred_m_slope_3h"] = future.groupby("station_id")["tide_pred_m"].diff(3)
        future["tide_percentile_station_year"] = future.groupby("station_id")["tide_pred_m"].rank(pct=True)
        atomic_parquet(future, output)
    return future

# Forecast stations are limited to mapped named beaches.
future_tides = collect_future_tides(beaches["tide_id"].dropna().astype(str).unique())
if future_tides.empty:
    raise RuntimeError("No future CO-OPS tide predictions were collected.")
print("Future tide rows:", f"{len(future_tides):,}")

## 19. Build a long-range wave and weather climatology

In [ ]:
CLIMATOLOGY_FEATURES = [
    "tide_obs_m",
    "surge_residual_m",
    "WVHT", "DPD", "APD", "MWD", "WSPD", "GST", "PRES", "ATMP", "WTMP",
    "wave_energy_proxy", "wave_steepness_proxy",
]

def _finite_numeric_values(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    return values[np.isfinite(values)]

def _safe_median(series: pd.Series) -> float:
    values = _finite_numeric_values(series)
    return float(np.median(values)) if values.size else np.nan

def _safe_quantile(series: pd.Series, quantile: float) -> float:
    values = _finite_numeric_values(series)
    return float(np.quantile(values, quantile)) if values.size else np.nan

def build_climatology(history: pd.DataFrame) -> pd.DataFrame:
    """
    Build station/month/hour climatology without invoking NumPy reductions on
    empty slices. Columns that have no finite observations anywhere in the
    historical panel are omitted and reported instead of producing hundreds of
    RuntimeWarning messages.
    """
    require_frame(history, "historical feature panel", ["station_id", "time"])
    history = history.copy()
    history["month"] = history["time"].dt.month
    history["hour"] = history["time"].dt.hour

    available_features = []
    dropped_features = []
    for col in CLIMATOLOGY_FEATURES:
        if col not in history:
            dropped_features.append(col)
            continue
        numeric = pd.to_numeric(history[col], errors="coerce")
        if np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).any():
            history[col] = numeric
            available_features.append(col)
        else:
            dropped_features.append(col)

    group_keys = ["station_id", "month", "hour"]
    if not available_features:
        climatology = history[group_keys].drop_duplicates().reset_index(drop=True)
        print(
            "No finite tide/wave climatology observations were available. "
            "Long-range simulation will use harmonic tides, zero surge residual, "
            "and model-side missing-value handling."
        )
    else:
        aggregations = {}
        for col in available_features:
            aggregations[f"{col}_median"] = (col, _safe_median)
            aggregations[f"{col}_p90"] = (
                col,
                lambda s, q=0.90: _safe_quantile(s, q),
            )
            aggregations[f"{col}_p10"] = (
                col,
                lambda s, q=0.10: _safe_quantile(s, q),
            )

        climatology = (
            history.groupby(group_keys, dropna=False)
            .agg(**aggregations)
            .reset_index()
        )

    climatology.attrs["available_features"] = available_features
    climatology.attrs["dropped_features"] = dropped_features
    atomic_parquet(climatology, PROCESSED / "station_month_hour_climatology.parquet")

    print(
        f"Climatology features available: {len(available_features)}/"
        f"{len(CLIMATOLOGY_FEATURES)}"
    )
    if dropped_features:
        print("Climatology features unavailable for this run:", dropped_features)
    return climatology

climatology = build_climatology(panel)
AVAILABLE_CLIMATOLOGY_FEATURES = list(
    climatology.attrs.get("available_features", [])
)
DROPPED_CLIMATOLOGY_FEATURES = list(
    climatology.attrs.get("dropped_features", [])
)
display(climatology.head())

## 20. Optional operational GFS Wave ingestion

This cell deliberately stops the operational wave tier at **16 days**. It downloads regional GRIB2 subsets from NCEP NOMADS. NOAA asks automated Grib Filter loops to wait at least 10 seconds between requests.

The exact variable names can change between GRIB editions, so the parser discovers candidate fields rather than assuming a single xarray name.

In [ ]:
NOMADS_GFSWAVE_FILTER = "https://nomads.ncep.noaa.gov/cgi-bin/filter_gfswave.pl"

def latest_available_gfswave_cycle() -> tuple[str, int]:
    # Search back up to three days and prefer the latest completed common cycle.
    now = datetime.now(timezone.utc)
    for day_offset in range(0, 4):
        date = (now - timedelta(days=day_offset)).strftime("%Y%m%d")
        for cycle in [18, 12, 6, 0]:
            # A small f000 request is used only when the operational cell is explicitly run.
            filename = f"gfswave.t{cycle:02d}z.global.0p25.f000.grib2"
            params = {
                "file": filename,
                "lev_surface": "on",
                "var_HTSGW": "on",
                "subregion": "",
                "leftlon": -130,
                "rightlon": -65,
                "toplat": 52,
                "bottomlat": 20,
                "dir": f"/gfs.{date}/{cycle:02d}/wave/gridded",
            }
            response = SESSION.get(NOMADS_GFSWAVE_FILTER, params=params, timeout=120)
            if response.ok and len(response.content) > 10_000 and b"GRIB" in response.content[:100]:
                return date, cycle
    raise RuntimeError("Could not locate a current GFS Wave cycle.")

def download_gfswave_subset(
    date: str,
    cycle: int,
    forecast_hour: int,
    *,
    force: bool = False,
) -> Path:
    output = RAW / "gfswave" / date / f"{cycle:02d}" / f"f{forecast_hour:03d}.grib2"
    if output.exists() and not force:
        return output
    output.parent.mkdir(parents=True, exist_ok=True)

    filename = f"gfswave.t{cycle:02d}z.global.0p25.f{forecast_hour:03d}.grib2"
    params = {
        "file": filename,
        "lev_surface": "on",
        "var_HTSGW": "on",
        "var_PERPW": "on",
        "var_DIRPW": "on",
        "var_WIND": "on",
        "var_WVDIR": "on",
        "subregion": "",
        "leftlon": -130,
        "rightlon": -65,
        "toplat": 52,
        "bottomlat": 20,
        "dir": f"/gfs.{date}/{cycle:02d}/wave/gridded",
    }
    content = get_bytes(NOMADS_GFSWAVE_FILTER, params=params, timeout=300)
    if b"GRIB" not in content[:100]:
        raise ValueError(f"NOMADS did not return GRIB2 for f{forecast_hour:03d}")
    output.write_bytes(content)
    return output

def open_grib_candidates(path: Path) -> list:
    try:
        import cfgrib
    except ImportError as exc:
        raise RuntimeError(
            "Operational GFS Wave ingestion requires optional packages cfgrib and eccodes. "
            "Re-run the installation cell or leave RUN_OPERATIONAL_GFSWAVE=False."
        ) from exc
    return cfgrib.open_datasets(str(path), backend_kwargs={"indexpath": ""})

def _pick_data_var(datasets: list, candidates: Sequence[str]):
    candidates = [c.lower() for c in candidates]
    for ds in datasets:
        for name, array in ds.data_vars.items():
            metadata = " ".join([
                name,
                str(array.attrs.get("GRIB_shortName", "")),
                str(array.attrs.get("long_name", "")),
                str(array.attrs.get("standard_name", "")),
            ]).lower()
            if any(candidate in metadata for candidate in candidates):
                return ds, name
    return None, None

def sample_grib_at_beaches(path: Path, beach_points: pd.DataFrame, valid_time: pd.Timestamp) -> pd.DataFrame:
    datasets = open_grib_candidates(path)
    ds_wave, wave_name = _pick_data_var(datasets, ["swh", "htsgw", "significant height"])
    ds_period, period_name = _pick_data_var(datasets, ["perpw", "primary wave period"])
    ds_direction, direction_name = _pick_data_var(datasets, ["dirpw", "primary wave direction"])

    if ds_wave is None:
        raise ValueError("Significant wave height field was not found in GRIB2.")

    rows = []
    for beach in beach_points.itertuples(index=False):
        select = {"latitude": float(beach.latitude), "longitude": float(beach.longitude)}
        # Some GFS longitudes use 0..360.
        if float(beach.longitude) < 0 and float(ds_wave.longitude.max()) > 180:
            select["longitude"] = float(beach.longitude) % 360

        wave_value = float(ds_wave[wave_name].sel(select, method="nearest").squeeze().values)
        period_value = (
            float(ds_period[period_name].sel(select, method="nearest").squeeze().values)
            if ds_period is not None else np.nan
        )
        direction_value = (
            float(ds_direction[direction_name].sel(select, method="nearest").squeeze().values)
            if ds_direction is not None else np.nan
        )
        rows.append({
            "beach_id": beach.beach_id,
            "time": valid_time,
            "WVHT": wave_value,
            "DPD": period_value,
            "MWD": direction_value,
            "wave_energy_proxy": wave_value ** 2 * period_value if pd.notna(period_value) else np.nan,
            "wave_steepness_proxy": wave_value / max(period_value ** 2, 0.25) if pd.notna(period_value) else np.nan,
            "wave_source": "NCEP GFS Wave",
        })
    return pd.DataFrame(rows)

def collect_operational_wave_forecast(
    beach_points: pd.DataFrame,
    max_hours: int = OPERATIONAL_WAVE_DAYS * 24,
    step_hours: int = 6,
) -> pd.DataFrame:
    date, cycle = latest_available_gfswave_cycle()
    initialization = pd.Timestamp(f"{date} {cycle:02d}:00:00", tz="UTC")
    forecast_hours = list(range(0, min(max_hours, 384) + 1, step_hours))
    # Demo mode keeps the download modest. Raise to 384 after validating the pipeline.
    if not FULL_SCALE:
        forecast_hours = [h for h in forecast_hours if h <= 72]

    frames = []
    for i, forecast_hour in enumerate(tqdm(forecast_hours, desc="GFS Wave forecast hours")):
        try:
            path = download_gfswave_subset(date, cycle, forecast_hour)
            frames.append(
                sample_grib_at_beaches(
                    path,
                    beach_points,
                    initialization + pd.Timedelta(hours=forecast_hour),
                )
            )
        except Exception as exc:
            print("GFS Wave hour failed:", forecast_hour, type(exc).__name__, str(exc)[:160])
        if i < len(forecast_hours) - 1:
            time.sleep(10.0)  # NOMADS responsible-use requirement.

    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not result.empty:
        atomic_parquet(result, FORECASTS / "operational_gfswave_beaches.parquet")
    return result

# Explicit opt-in because a national GRIB loop is network- and storage-intensive.
RUN_OPERATIONAL_GFSWAVE = False
operational_wave = (
    collect_operational_wave_forecast(
        beaches[["beach_id", "latitude", "longitude"]].drop_duplicates("beach_id")
    )
    if RUN_OPERATIONAL_GFSWAVE
    else pd.DataFrame()
)
print("Operational wave rows:", len(operational_wave))

## 21. Monte Carlo 300-day beach risk surface

In [ ]:
def prepare_future_feature_rows(
    future_tides: pd.DataFrame,
    beaches: pd.DataFrame,
    climatology: pd.DataFrame,
) -> pd.DataFrame:
    # One tide station may serve multiple beaches.
    future = beaches[[
        "beach_id", "beach_name", "state", "county", "latitude", "longitude",
        "tide_id", "tide_distance_km",
    ]].merge(
        future_tides,
        left_on="tide_id",
        right_on="station_id",
        how="inner",
    )
    future["month"] = future["time"].dt.month
    future["hour"] = future["time"].dt.hour
    future["dayofyear"] = future["time"].dt.dayofyear
    future["weekday"] = future["time"].dt.weekday
    future["is_weekend"] = (future["weekday"] >= 5).astype(int)
    future["hour_sin"] = np.sin(2 * np.pi * future["hour"] / 24)
    future["hour_cos"] = np.cos(2 * np.pi * future["hour"] / 24)
    future["doy_sin"] = np.sin(2 * np.pi * future["dayofyear"] / 365.25)
    future["doy_cos"] = np.cos(2 * np.pi * future["dayofyear"] / 365.25)
    future["daylight_proxy"] = future["hour"].between(7, 19).astype(int)
    future["warm_season_proxy"] = future["month"].between(5, 9).astype(int)
    future["crowd_proxy"] = (
        0.4 * future["is_weekend"]
        + 0.35 * future["daylight_proxy"]
        + 0.25 * future["warm_season_proxy"]
    )

    future = future.merge(
        climatology,
        on=["station_id", "month", "hour"],
        how="left",
    )
    station_meta = coops.set_index("station_id")
    future["station_latitude"] = future["station_id"].map(station_meta["latitude"])
    future["station_longitude"] = future["station_id"].map(station_meta["longitude"])
    return future

def monte_carlo_future_risk(
    future: pd.DataFrame,
    draws: int = 30,
    chunk_size: int = 150_000,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Draw uncertain future wave/weather states from station-month-hour climatological
    quantiles, preserving exact future harmonic tides. Returns risk mean/p90/spread.
    """
    if draws < 1:
        raise ValueError("draws must be at least 1")
    if future.empty:
        return pd.DataFrame()
    output_frames = []
    local_rng = np.random.default_rng(seed)
    now = pd.Timestamp.now(tz="UTC")

    for start in tqdm(range(0, len(future), chunk_size), desc="Monte Carlo risk chunks"):
        chunk = future.iloc[start:start + chunk_size].copy()
        risk_draws = []

        for draw in range(draws):
            synthetic = chunk.copy()
            for feature in AVAILABLE_CLIMATOLOGY_FEATURES:
                median = synthetic.get(f"{feature}_median", pd.Series(np.nan, index=synthetic.index))
                low = synthetic.get(f"{feature}_p10", median)
                high = synthetic.get(f"{feature}_p90", median)
                # Truncated normal-like interpolation inside historical P10/P90.
                z = np.clip(local_rng.normal(0.0, 0.55, len(synthetic)), -1.0, 1.0)
                values = np.where(
                    z >= 0,
                    median + z * (high - median),
                    median + (-z) * (low - median),
                )
                synthetic[feature] = values

            # Future observed water level is unknown. When historical surge
            # observations are unavailable, use a neutral zero-residual fallback
            # rather than propagating an all-NaN climatology through every draw.
            if "surge_residual_m" in synthetic:
                surge_residual = pd.to_numeric(
                    synthetic["surge_residual_m"], errors="coerce"
                ).fillna(0.0)
            else:
                surge_residual = pd.Series(0.0, index=synthetic.index)
            synthetic["surge_residual_m"] = surge_residual
            synthetic["tide_obs_m"] = (
                pd.to_numeric(synthetic["tide_pred_m"], errors="coerce")
                + surge_residual
            )
            synthetic["tide_obs_m_slope_1h"] = synthetic.groupby("station_id")["tide_obs_m"].diff(1)
            synthetic["tide_obs_m_slope_3h"] = synthetic.groupby("station_id")["tide_obs_m"].diff(3)
            synthetic["surge_residual_m_slope_1h"] = synthetic.groupby("station_id")["surge_residual_m"].diff(1)
            synthetic["surge_residual_m_slope_3h"] = synthetic.groupby("station_id")["surge_residual_m"].diff(3)

            for c in FEATURE_COLUMNS:
                if c not in synthetic:
                    synthetic[c] = np.nan

            supervised_features = list(model_bundle.get("features", FEATURE_COLUMNS))
            raw, _ = ensemble_predict(
                model_bundle["ensemble"], synthetic, supervised_features
            )
            probability = (
                model_bundle["calibrator"].predict(raw)
                if model_bundle["calibrator"] is not None
                else raw
            )
            risk_draws.append(probability)

        matrix = np.vstack(risk_draws)
        result = chunk[[
            "beach_id", "beach_name", "state", "county", "latitude", "longitude",
            "station_id", "time", "tide_pred_m", "tide_pred_m_slope_3h",
            "tide_percentile_station_year", "crowd_proxy",
        ]].copy()
        result["risk_mean"] = matrix.mean(axis=0)
        result["risk_p90"] = np.quantile(matrix, 0.90, axis=0)
        result["simulation_spread"] = matrix.std(axis=0)
        result["lead_days"] = (result["time"] - now).dt.total_seconds() / 86400.0
        result["forecast_tier"] = np.where(
            result["lead_days"] <= OPERATIONAL_WAVE_DAYS,
            "near-term-awaiting-wave-overlay",
            "long-range-climatology",
        )
        # Long-range values get an explicit uncertainty penalty.
        result["uncertainty"] = (
            result["simulation_spread"]
            + np.clip((result["lead_days"] - OPERATIONAL_WAVE_DAYS) / FORECAST_DAYS, 0, 1) * 0.20
        ).clip(0, 1)
        output_frames.append(result)

    return pd.concat(output_frames, ignore_index=True)

future_features = prepare_future_feature_rows(future_tides, beaches, climatology)
require_frame(future_features, "future beach/tide feature rows", ["beach_id", "station_id", "time"])
future_risk = monte_carlo_future_risk(
    future_features,
    draws=15 if not FULL_SCALE else 50,
)
atomic_parquet(future_risk, FORECASTS / "beach_risk_300d_raw.parquet")

print("Forecast beach-hours:", f"{len(future_risk):,}")
display(future_risk.sort_values("risk_p90", ascending=False).head(20))

## 22. Overlay operational waves and assign alert actions

In [ ]:
def overlay_operational_wave(
    risk: pd.DataFrame,
    operational_wave: pd.DataFrame,
) -> pd.DataFrame:
    out = risk.copy()
    if operational_wave.empty:
        out["wave_overlay_used"] = False
        return out

    wave = operational_wave.sort_values(["beach_id", "time"])
    groups = []
    for beach_id, part in out.groupby("beach_id", sort=False):
        beach_wave = wave[wave["beach_id"] == beach_id]
        if beach_wave.empty:
            part["wave_overlay_used"] = False
            groups.append(part)
            continue
        merged = pd.merge_asof(
            part.sort_values("time"),
            beach_wave.sort_values("time"),
            on="time",
            by="beach_id",
            direction="nearest",
            tolerance=pd.Timedelta(hours=4),
            suffixes=("", "_operational"),
        )
        merged["wave_overlay_used"] = merged["WVHT"].notna()
        # Transparent monotonic overlay: high energy can only raise near-term concern.
        energy_boost = np.clip(
            np.nan_to_num(merged["wave_energy_proxy"], nan=0.0) / 40.0,
            0,
            0.35,
        )
        merged["risk_mean"] = np.maximum(
            merged["risk_mean"],
            1 - (1 - merged["risk_mean"]) * (1 - energy_boost),
        )
        merged["risk_p90"] = np.maximum(merged["risk_p90"], merged["risk_mean"])
        merged.loc[merged["wave_overlay_used"], "forecast_tier"] = "operational-tide+gfswave"
        groups.append(merged)
    return pd.concat(groups, ignore_index=True)

forecast = overlay_operational_wave(future_risk, operational_wave)

def choose_alert_action(row) -> int:
    # Use the calibrated deterministic fallback even when an RL checkpoint exists.
    # Production deployments should load/evaluate the PPO policy and compare it
    # against this safety baseline before enabling it.
    score = float(max(row.risk_mean, row.risk_p90 - 0.5 * row.uncertainty))
    if score >= model_bundle["strong_threshold"]:
        return 2
    if score >= model_bundle["alert_threshold"]:
        return 1
    return 0

forecast["alert_action"] = forecast.apply(choose_alert_action, axis=1)
forecast["alert_name"] = forecast["alert_action"].map({
    0: "normal-monitoring",
    1: "advisory-review",
    2: "strong-warning-review",
})

# Avoid presenting every hourly row. Merge consecutive alert hours into practical windows.
def collapse_alert_windows(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("time").copy()
    group = group[group["alert_action"] > 0]
    if group.empty:
        return pd.DataFrame()
    new_window = (
        group["time"].diff().gt(pd.Timedelta(hours=2))
        | group["alert_action"].ne(group["alert_action"].shift())
    )
    group["window_id"] = new_window.cumsum()
    return (
        group.groupby("window_id")
        .agg(
            beach_id=("beach_id", "first"),
            beach_name=("beach_name", "first"),
            state=("state", "first"),
            county=("county", "first"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
            start_time=("time", "min"),
            end_time=("time", "max"),
            alert_action=("alert_action", "max"),
            alert_name=("alert_name", "last"),
            risk_mean=("risk_mean", "max"),
            risk_p90=("risk_p90", "max"),
            uncertainty=("uncertainty", "max"),
            tide_peak_m=("tide_pred_m", "max"),
            forecast_tier=("forecast_tier", "last"),
        )
        .reset_index(drop=True)
    )

window_frames = [
    collapse_alert_windows(group)
    for _, group in forecast.groupby("beach_id", sort=False)
]
alert_windows = pd.concat([x for x in window_frames if not x.empty], ignore_index=True) if any(
    not x.empty for x in window_frames
) else pd.DataFrame()

if not alert_windows.empty:
    alert_windows = alert_windows.sort_values(
        ["alert_action", "risk_p90", "start_time"],
        ascending=[False, False, True],
    )
    alert_windows.to_csv(FORECASTS / "beach_alert_windows_300d.csv", index=False)
    atomic_parquet(alert_windows, FORECASTS / "beach_alert_windows_300d.parquet")

atomic_parquet(forecast, FORECASTS / "beach_risk_300d_scored.parquet")
display(alert_windows.head(50) if not alert_windows.empty else "No alert windows at current thresholds.")

## 23. Historical and forecast hot-zone map

In [ ]:
import folium
from folium.plugins import HeatMap, MarkerCluster

historical_hotspots = (
    events.groupby(["tide_id"], dropna=True)
    .agg(
        historical_events=("event_id", "nunique"),
        historical_severity=("severity", "sum"),
        historical_rescues_weak=("rescued_count_weak", "sum"),
    )
    .reset_index()
)

beach_hotspots = beaches.merge(
    historical_hotspots,
    left_on="tide_id",
    right_on="tide_id",
    how="left",
)
beach_hotspots[["historical_events", "historical_severity", "historical_rescues_weak"]] = (
    beach_hotspots[["historical_events", "historical_severity", "historical_rescues_weak"]].fillna(0)
)

future_hotspots = (
    forecast.groupby("beach_id")
    .agg(
        max_future_risk=("risk_p90", "max"),
        future_alert_hours=("alert_action", lambda s: int((s > 0).sum())),
        future_strong_hours=("alert_action", lambda s: int((s == 2).sum())),
    )
    .reset_index()
)
beach_hotspots = beach_hotspots.merge(future_hotspots, on="beach_id", how="left").fillna({
    "max_future_risk": 0,
    "future_alert_hours": 0,
    "future_strong_hours": 0,
})
beach_hotspots["hot_zone_score"] = (
    np.log1p(beach_hotspots["historical_severity"])
    + 4 * beach_hotspots["max_future_risk"]
    + np.log1p(beach_hotspots["future_alert_hours"])
)

require_frame(beach_hotspots, "beach hot-zone table", ["latitude", "longitude", "hot_zone_score"])
center = [float(beach_hotspots["latitude"].median()), float(beach_hotspots["longitude"].median())]
hazard_map = folium.Map(location=center, zoom_start=4, tiles="CartoDB positron")

heat_rows = [
    [row.latitude, row.longitude, max(0.01, float(row.hot_zone_score))]
    for row in beach_hotspots.itertuples(index=False)
    if pd.notna(row.latitude) and pd.notna(row.longitude)
]
HeatMap(heat_rows, radius=10, blur=14, min_opacity=0.20).add_to(hazard_map)

cluster = MarkerCluster(name="Named beach risk").add_to(hazard_map)
for row in beach_hotspots.nlargest(min(500, len(beach_hotspots)), "hot_zone_score").itertuples(index=False):
    popup = folium.Popup(
        f"""
        <b>{row.beach_name}, {row.state}</b><br>
        GPS: {row.latitude:.5f}, {row.longitude:.5f}<br>
        Historical events: {int(row.historical_events)}<br>
        Historical severity: {row.historical_severity:.1f}<br>
        Max future P90 risk: {row.max_future_risk:.3f}<br>
        Forecast alert hours: {int(row.future_alert_hours)}<br>
        Tide station distance: {row.tide_distance_km:.1f} km
        """,
        max_width=350,
    )
    folium.Marker([row.latitude, row.longitude], popup=popup).add_to(cluster)

folium.LayerControl().add_to(hazard_map)
map_path = REPORTS / "surfguard_hot_zones.html"
hazard_map.save(map_path)

beach_hotspots.to_csv(REPORTS / "beach_hot_zone_scores.csv", index=False)
display(hazard_map)
print("Saved:", map_path)

## 24. OpenAI Responses API grounded warning composer

In [ ]:
from getpass import getpass

def get_openai_client():
    from openai import OpenAI
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            key = None
    if not key:
        key = getpass("OPENAI_API_KEY: ").strip()
    if not key:
        raise RuntimeError("OPENAI_API_KEY was not supplied.")
    return OpenAI(api_key=key)

def compose_grounded_warning(record: dict, client=None) -> str:
    client = client or get_openai_client()

    evidence = {
        "beach_name": record.get("beach_name"),
        "state": record.get("state"),
        "latitude": round(float(record.get("latitude")), 5),
        "longitude": round(float(record.get("longitude")), 5),
        "start_time_utc": str(record.get("start_time")),
        "end_time_utc": str(record.get("end_time")),
        "risk_mean": round(float(record.get("risk_mean")), 4),
        "risk_p90": round(float(record.get("risk_p90")), 4),
        "uncertainty": round(float(record.get("uncertainty")), 4),
        "tide_peak_m": round(float(record.get("tide_peak_m")), 3),
        "forecast_tier": record.get("forecast_tier"),
        "alert_name": record.get("alert_name"),
    }

    instructions = """
You are the communications layer for a coastal-safety research system.
Use only the supplied JSON evidence. Never invent wave heights, rip-current
observations, rescue counts, closures, official alerts, or certainty.

Write one public-address warning of 45-85 words:
1. Say the beach name and UTC time window.
2. State whether the result is operational or long-range climatological.
3. Describe the risk as a model-estimated hazardous-condition probability,
   not a prediction that a named person will be rescued.
4. Give conservative actions: stay near lifeguards, obey flags/closures,
   avoid entering rough water, and check local NOAA/NWS/lifeguard updates.
5. Read the GPS coordinates naturally.
6. Mention elevated uncertainty when uncertainty >= 0.25.
Do not claim this system is an official warning authority.
"""

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {"role": "developer", "content": instructions},
            {"role": "user", "content": json.dumps(evidence)},
        ],
    )
    return response.output_text.strip()

# Generate only a small reviewed batch by default to control cost.
RUN_GPT_WARNINGS = False
gpt_warning_rows = []

if RUN_GPT_WARNINGS and not alert_windows.empty:
    client = get_openai_client()
    for row in alert_windows.head(10).to_dict(orient="records"):
        text = compose_grounded_warning(row, client=client)
        row["warning_text"] = text
        gpt_warning_rows.append(row)
        print("\n", text)

    pd.DataFrame(gpt_warning_rows).to_json(
        REPORTS / "openai_grounded_warnings.jsonl",
        orient="records",
        lines=True,
        force_ascii=False,
    )

## 25. Audio beach warnings

In [ ]:
def synthesize_warning_audio(
    text: str,
    output_path: str | Path,
    client=None,
    *,
    model: str = OPENAI_TTS_MODEL,
    voice: str = OPENAI_TTS_VOICE,
) -> Path:
    client = client or get_openai_client()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with client.audio.speech.with_streaming_response.create(
        model=model,
        voice=voice,
        input=text,
        response_format="mp3",
    ) as response:
        response.stream_to_file(output_path)
    return output_path

RUN_AUDIO = False
if RUN_AUDIO and gpt_warning_rows:
    client = get_openai_client()
    for i, row in enumerate(gpt_warning_rows):
        safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", row["beach_name"]).strip("_")
        path = AUDIO / f"{i:03d}_{safe_name}.mp3"
        synthesize_warning_audio(row["warning_text"], path, client=client)
        print(path)

## 26. Optional structured OpenAI incident curation from historical narratives

In [ ]:
def curate_incident_with_gpt(event_record: dict, client=None) -> dict:
    """
    Optional expensive curation layer. It extracts structured facts only from the
    supplied narrative. Its output must remain a secondary annotation, never a
    replacement for the original report.
    """
    client = client or get_openai_client()
    payload = {
        "event_id": str(event_record.get("event_id")),
        "event_type": str(event_record.get("event_type")),
        "event_time": str(event_record.get("event_time")),
        "location_name": str(event_record.get("location_name")),
        "narrative": str(event_record.get("narrative", ""))[:12_000],
    }
    prompt = """
Return JSON only with keys:
event_id, rescue_count, rip_current_mentioned, shorebreak_mentioned,
large_swell_mentioned, people_entered_to_help, confidence, evidence_excerpt.

Rules:
- Extract only explicitly supported facts.
- rescue_count is null when not stated.
- evidence_excerpt is at most 20 words copied from the supplied narrative.
- confidence is a number from 0 to 1.
- Never infer a rescue count from vague wording.
"""
    response = client.responses.create(
        model=OPENAI_MODEL,
        text={"format": {
            "type": "json_schema",
            "name": "surfguard_incident_curation",
            "strict": True,
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "event_id": {"type": "string"},
                    "rescue_count": {"type": ["integer", "null"], "minimum": 0},
                    "rip_current_mentioned": {"type": "boolean"},
                    "shorebreak_mentioned": {"type": "boolean"},
                    "large_swell_mentioned": {"type": "boolean"},
                    "people_entered_to_help": {"type": "boolean"},
                    "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                    "evidence_excerpt": {"type": "string"}
                },
                "required": [
                    "event_id", "rescue_count", "rip_current_mentioned",
                    "shorebreak_mentioned", "large_swell_mentioned",
                    "people_entered_to_help", "confidence", "evidence_excerpt"
                ]
            }
        }},
        input=[
            {"role": "developer", "content": prompt},
            {"role": "user", "content": json.dumps(payload)},
        ],
    )
    text = response.output_text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    return json.loads(text)

RUN_GPT_INCIDENT_CURATION = False
if RUN_GPT_INCIDENT_CURATION:
    client = get_openai_client()
    curated = []
    for record in events[events["narrative"].str.len() > 40].head(25).to_dict(orient="records"):
        try:
            curated.append(curate_incident_with_gpt(record, client=client))
        except Exception as exc:
            print("Curation failed:", record["event_id"], type(exc).__name__)
    pd.DataFrame(curated).to_json(
        PROCESSED / "openai_incident_annotations.jsonl",
        orient="records",
        lines=True,
    )
    display(pd.DataFrame(curated).head())

## 27. Export model card, data card, checksums, and deployment bundle

In [ ]:
def dataframe_time_range(frame: pd.DataFrame, column: str = "time") -> dict:
    if frame.empty or column not in frame:
        return {"min": None, "max": None}
    values = frame[column].dropna()
    if values.empty:
        return {"min": None, "max": None}
    return {"min": str(values.min()), "max": str(values.max())}

model_card = {
    "name": "SurfGuard-USA hazard ensemble",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "geographic_scope": "Ocean-facing contiguous United States only",
    "excluded_geographies": ["Alaska", "Hawaii", "Puerto Rico", "U.S. territories", "Great Lakes"],
    "target": "Historically dangerous coastal-event window near a tide station",
    "not_a_target": "Exact future individual rescue or official closure decision",
    "requested_features": FEATURE_COLUMNS,
    "active_features": list(model_bundle.get("features", FEATURE_COLUMNS)),
    "dropped_all_missing_features": list(model_bundle.get("dropped_all_missing_features", [])),
    "split_summary": metrics.get("split_summary", {}),
    "training_rows": int(len(dataset)),
    "positive_rows": int(dataset["danger_label"].sum()),
    "historical_panel_time": dataframe_time_range(labeled_panel),
    "metrics": metrics,
    "alert_threshold": float(model_bundle["alert_threshold"]),
    "strong_threshold": float(model_bundle["strong_threshold"]),
    "forecast_horizons": {
        "0_to_16_days": "Operational tier only after current wave overlay; tides alone are insufficient",
        "17_to_300_days": "Astronomical tides plus climatological wave/hazard simulation",
    },
    "known_limitations": [
        "NOAA Storm Events does not enumerate every lifeguard rescue.",
        "Location coordinates may identify a zone rather than the exact incident point.",
        "Buoys and tide stations can be far from individual beaches.",
        "Nearshore bathymetry, sandbar migration, wind, swell angle, and beach morphology may be missing.",
        "Crowd exposure uses proxies unless local attendance data is supplied.",
        "Long-range wave states are climatological scenarios, not deterministic forecasts.",
        "Public warnings require human and official-source review.",
    ],
}
(REPORTS / "MODEL_CARD.json").write_text(json.dumps(json_safe(model_card), indent=2, allow_nan=False), encoding="utf-8")

data_card = {
    "sources": SOURCE_MANIFEST.to_dict(orient="records"),
    "events_rows": int(len(events)),
    "beaches_rows": int(len(beaches)),
    "coops_rows": int(len(coops)),
    "ndbc_rows": int(len(ndbc)),
    "panel_rows": int(len(panel)),
    "forecast_rows": int(len(forecast)),
}
(REPORTS / "DATA_CARD.json").write_text(json.dumps(json_safe(data_card), indent=2, allow_nan=False), encoding="utf-8")

artifacts = [
    MODELS / "surfguard_supervised_bundle.joblib",
    REPORTS / "MODEL_CARD.json",
    REPORTS / "DATA_CARD.json",
    REPORTS / "supervised_metrics.json",
    FORECASTS / "beach_risk_300d_scored.parquet",
    REPORTS / "surfguard_hot_zones.html",
]
checksums = []
for path in artifacts:
    if path.exists():
        checksums.append({
            "path": str(path.relative_to(ROOT)),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
pd.DataFrame(checksums).to_csv(REPORTS / "checksums.csv", index=False)

bundle_stage = ROOT / "deployment_bundle_stage"
if bundle_stage.exists():
    shutil.rmtree(bundle_stage)
bundle_stage.mkdir(parents=True)
for path in artifacts:
    if path.exists():
        relative = path.relative_to(ROOT)
        target = bundle_stage / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
shutil.copy2(REPORTS / "checksums.csv", bundle_stage / "checksums.csv")
(bundle_stage / "README.txt").write_text(
    "SurfGuard-USA research artifacts. Not an official NOAA/NWS warning product.\n",
    encoding="utf-8",
)
bundle_path = shutil.make_archive(
    str(ROOT / "surfguard_deployment_bundle"),
    "zip",
    root_dir=bundle_stage,
)
print("Deployment bundle:", bundle_path)
display(pd.DataFrame(checksums))

## 28. Production hardening checklist

Before any real beach deployment:

1. **Ingest local lifeguard dispatch logs.** National storm reports are useful but incomplete for rescue demand.
2. **Add beach morphology.** Nearshore bathymetry, slope, sandbars, headlands, jetties, and swell exposure strongly affect local hazards.
3. **Use real attendance/exposure data.** A dangerous ocean with nobody present is different from a crowded holiday afternoon.
4. **Backtest by geography and year.** Hold out entire beaches, counties, storm seasons, and years—not only random rows.
5. **Calibrate locally.** Thresholds for Santa Cruz should not be copied blindly to the Outer Banks or Gulf Coast.
6. **Run shadow mode.** Compare recommendations against official alerts and lifeguard judgment without publishing automated warnings.
7. **Add drift monitoring.** Track sensor outages, missing features, station distance, calibration decay, and false-alarm burden.
8. **Human approval remains mandatory.** The system should draft warnings and rank hotspots; qualified local authorities decide what is announced.
9. **Never describe 300-day output as exact surf prediction.** It is a tide-informed seasonal planning-risk calendar.
10. **Respect source services.** Cache data, partition by station/year, retry safely, and rate-limit all NOAA requests.

## 29. Minimal run order

In [ ]:
RUN_ORDER = [
    "1. Install dependencies and PufferForge",
    "2. Configure FULL_SCALE and storage",
    "3. Collect NOAA Storm Events",
    "4. Add optional verified local rescue logs",
    "5. Collect GNIS beaches, CO-OPS stations, and NDBC stations",
    "6. Select station-years and collect tide/buoy history",
    "7. Build and label the hourly panel",
    "8. Train/calibrate the supervised ensemble",
    "9. Train PufferForge PPO alert policy",
    "10. Collect 300-day future tides",
    "11. Optionally enable near-term GFS Wave",
    "12. Run Monte Carlo risk surface and collapse alert windows",
    "13. Review maps and metrics",
    "14. Manually enable GPT warnings/audio for selected reviewed windows",
]
for step in RUN_ORDER:
    print(step)